In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22

# =========================
# LOAD BOTH SHEETS
# =========================

hz = pd.read_excel("input.xlsx", sheet_name="HZ")
vt = pd.read_excel("input.xlsx", sheet_name="VT")

# =========================
# FUNCTION TO MELT MACHINES
# =========================

def expand_machines(df):

    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_long = df.melt(
        id_vars=["Part","Inventory","Monthly_Indent","Category","Cycle time","Cavity"],
        value_vars=machine_cols,
        var_name="Machine_Col",
        value_name="Machine"
    )

    df_long = df_long.dropna(subset=["Machine"])

    return df_long

hz_long = expand_machines(hz)
vt_long = expand_machines(vt)

df = pd.concat([hz_long, vt_long], ignore_index=True)

# =========================
# CALCULATE RATE
# =========================

df["Rate"] = (3600 / df["Cycle time"]) * df["Cavity"]

# =========================
# MRP LOGIC
# =========================

df["Daily_Demand"] = df["Monthly_Indent"] / WORKING_DAYS
df["Safety"] = df["Daily_Demand"] * 3
df["Gap"] = df["Safety"] - df["Inventory"]
df["Trigger"] = np.where(df["Gap"] > 0, 1, 0)

# =========================
# CATEGORY TARGET
# =========================

def target_days(cat):
    if cat == "Runner":
        return 5
    elif cat == "Repeater":
        return 3
    else:
        return 1

df["Target_Days"] = df["Category"].apply(target_days)
df["Target_Inventory"] = df["Daily_Demand"] * df["Target_Days"]

# =========================
# PRODUCTION QTY
# =========================

df["Production_Qty"] = np.where(
    df["Trigger"] == 1,
    df["Target_Inventory"] - df["Inventory"],
    0
)

df["Production_Qty"] = df["Production_Qty"].clip(lower=0)

# =========================
# HOURS REQUIRED
# =========================

df["Hours_Required"] = df["Production_Qty"] / df["Rate"]

# =========================
# PRIORITY
# =========================

priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
df["Priority"] = df["Category"].map(priority_map)

df = df.sort_values(by="Priority", ascending=False)

# =========================
# ASSIGN BEST MACHINE
# =========================

df = df.sort_values(by="Hours_Required")

df = df.drop_duplicates(subset=["Part"], keep="first")

# =========================
# MACHINE LOAD CHECK
# =========================

machine_load = df.groupby("Machine")["Hours_Required"].sum().reset_index()
machine_load["Overload"] = machine_load["Hours_Required"] - MACHINE_CAPACITY

# =========================
# HANDLE OVERLOAD
# =========================

for machine in machine_load["Machine"]:

    overload = machine_load.loc[machine_load["Machine"] == machine, "Overload"].values[0]

    if overload > 0:

        subset = df[df["Machine"] == machine]

        strangers = subset[subset["Category"]=="Stranger"]

        for idx in strangers.index:
            if overload <= 0:
                break
            overload -= df.loc[idx,"Hours_Required"]
            df.loc[idx,"Production_Qty"] = 0
            df.loc[idx,"Hours_Required"] = 0

        repeaters = subset[subset["Category"]=="Repeater"]

        for idx in repeaters.index:
            if overload <= 0:
                break
            overload -= df.loc[idx,"Hours_Required"]
            df.loc[idx,"Production_Qty"] = 0
            df.loc[idx,"Hours_Required"] = 0

# =========================
# FINAL OUTPUT
# =========================

df["Produce_Today"] = np.where(df["Production_Qty"]>0,"YES","NO")

df_final = df[[
    "Part",
    "Category",
    "Inventory",
    "Production_Qty",
    "Machine",
    "Hours_Required",
    "Produce_Today"
]]

df_final.to_excel("APS_Plan.xlsx", index=False)

print("APS Plan Generated")

In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22

# =========================
# LOAD BOTH SHEETS
# =========================

file_path = "C:/Users/Ex0164/Book1.xlsx"

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")

# =========================
# FUNCTION TO EXPAND MACHINES
# =========================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Category","Cycle time","Cavity"]

    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]

        temp = temp[temp["Machine"].notna()]
        temp = temp[temp["Machine"] != ""]

        df_list.append(temp)

    return pd.concat(df_list, ignore_index=True)

# Expand both sheets
hz_long = expand_machines(hz)
vt_long = expand_machines(vt)

df = pd.concat([hz_long, vt_long], ignore_index=True)

# =========================
# CALCULATE RATE
# =========================

df["Rate"] = (3600 / df["Cycle time"]) * df["Cavity"]

# =========================
# MRP LOGIC
# =========================

df["Daily_Demand"] = df["Indent"] / WORKING_DAYS
df["Safety"] = df["Daily_Demand"] * 3
df["Gap"] = df["Safety"] - df["Inventory"]
df["Trigger"] = np.where(df["Gap"] > 0, 1, 0)

# =========================
# CATEGORY TARGET
# =========================

def target_days(cat):
    if cat == "Runner":
        return 5
    elif cat == "Repeater":
        return 3
    else:
        return 1

df["Target_Days"] = df["Category"].apply(target_days)
df["Target_Inventory"] = df["Daily_Demand"] * df["Target_Days"]

# =========================
# PRODUCTION QTY
# =========================

df["Production_Qty"] = np.where(
    df["Trigger"] == 1,
    df["Target_Inventory"] - df["Inventory"],
    0
)

df["Production_Qty"] = df["Production_Qty"].clip(lower=0)

# =========================
# HOURS REQUIRED
# =========================

df["Hours_Required"] = df["Production_Qty"] / df["Rate"]

# =========================
# PRIORITY
# =========================

priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
df["Priority"] = df["Category"].map(priority_map)

df = df.sort_values(by=["Priority","Hours_Required"], ascending=[False,True])

# =========================
# SELECT BEST MACHINE
# =========================

df = df.drop_duplicates(subset=["Part"], keep="first")

# =========================
# MACHINE LOAD CHECK
# =========================

machine_load = df.groupby("Machine")["Hours_Required"].sum().reset_index()
machine_load["Overload"] = machine_load["Hours_Required"] - MACHINE_CAPACITY

# =========================
# HANDLE OVERLOAD
# =========================

for machine in machine_load["Machine"]:

    overload = machine_load.loc[machine_load["Machine"] == machine, "Overload"].values[0]

    if overload > 0:

        subset = df[df["Machine"] == machine]

        # Remove Stranger first
        strangers = subset[subset["Category"]=="Stranger"]

        for idx in strangers.index:
            if overload <= 0:
                break
            overload -= df.loc[idx,"Hours_Required"]
            df.loc[idx,"Production_Qty"] = 0
            df.loc[idx,"Hours_Required"] = 0

        # Then Repeater
        repeaters = subset[subset["Category"]=="Repeater"]

        for idx in repeaters.index:
            if overload <= 0:
                break
            overload -= df.loc[idx,"Hours_Required"]
            df.loc[idx,"Production_Qty"] = 0
            df.loc[idx,"Hours_Required"] = 0

# =========================
# FINAL OUTPUT
# =========================

df["Produce_Today"] = np.where(df["Production_Qty"]>0,"YES","NO")

df_final = df[[
    "Part",
    "Category",
    "Inventory",
    "Production_Qty",
    "Machine",
    "Hours_Required",
    "Produce_Today"
]]

df_final.to_excel("APS_Plan.xlsx", index=False)

print("APS Plan Generated Successfully")

In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22
MIN_RUN_HOURS = 4

file_path = "C:/Users/Ex0164/Book1.xlsx"

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")

# =========================
# EXPAND MACHINES
# =========================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Category","Cycle time","Cavity"]
    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]

        temp = temp[temp["Machine"].notna()]
        temp = temp[temp["Machine"] != ""]

        df_list.append(temp)

    return pd.concat(df_list, ignore_index=True)

# =========================
# APS LOGIC FUNCTION
# =========================

def run_aps(df):

    df["Rate"] = (3600 / df["Cycle time"]) * df["Cavity"]

    df["Daily_Demand"] = df["Indent"] / WORKING_DAYS
    df["Safety"] = df["Daily_Demand"] * 3
    df["Gap"] = df["Safety"] - df["Inventory"]

    df["Trigger"] = np.where(df["Gap"] > 0, 1, 0)

    def target_days(cat):
        if cat == "Runner":
            return 5
        elif cat == "Repeater":
            return 3
        else:
            return 1

    df["Target_Days"] = df["Category"].apply(target_days)
    df["Target_Inventory"] = df["Daily_Demand"] * df["Target_Days"]

    df["Production_Qty"] = np.where(
        df["Trigger"] == 1,
        df["Target_Inventory"] - df["Inventory"],
        0
    )

    df["Production_Qty"] = df["Production_Qty"].clip(lower=0)

    df["Hours_Required"] = df["Production_Qty"] / df["Rate"]

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
    df["Priority"] = df["Category"].map(priority_map)

    df = df.sort_values(by=["Priority","Hours_Required"], ascending=[False,True])

    # =========================
    # MACHINE-WISE ALLOCATION
    # =========================

    final_plan = []

    for machine, group in df.groupby("Machine"):

        remaining_hours = MACHINE_CAPACITY

        for _, row in group.iterrows():

            if remaining_hours <= 0:
                break

            hours = row["Hours_Required"]

            if hours < MIN_RUN_HOURS:
                hours = MIN_RUN_HOURS

            if hours > remaining_hours:
                continue

            qty = hours * row["Rate"]

            final_plan.append({
                "Part": row["Part"],
                "Category": row["Category"],
                "Machine": machine,
                "Run_Hours": hours,
                "Production_Qty": qty
            })

            remaining_hours -= hours

    final_df = pd.DataFrame(final_plan)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df, machine_summary

# =========================
# RUN APS FOR BOTH
# =========================

hz_long = expand_machines(hz)
vt_long = expand_machines(vt)

hz_plan, hz_machine = run_aps(hz_long)
vt_plan, vt_machine = run_aps(vt_long)

# =========================
# SAVE OUTPUT
# =========================

with pd.ExcelWriter("APS_Plan.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

print("APS Plan Generated with Machine-wise Allocation")

In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22
MIN_RUN_HOURS = 4

file_path = "C:/Users/Ex0164/Book1.xlsx"

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")

# =========================
# EXPAND MACHINES
# =========================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Category","Cycle time","Cavity"]
    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]

        temp = temp[temp["Machine"].notna()]
        temp = temp[temp["Machine"] != ""]

        df_list.append(temp)

    return pd.concat(df_list, ignore_index=True)

# =========================
# SMART APS ENGINE
# =========================

def run_smart_aps(df):

    df["Rate"] = (3600 / df["Cycle time"]) * df["Cavity"]
    df["Daily_Demand"] = df["Indent"] / WORKING_DAYS
    df["Safety"] = df["Daily_Demand"] * 3

    # Current coverage
    df["Coverage"] = df["Inventory"] / df["Daily_Demand"]

    # Risk level
    df["Risk"] = 3 - df["Coverage"]

    df = df[df["Risk"] > 0].copy()

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
    df["Priority"] = df["Category"].map(priority_map)

    df = df.sort_values(by=["Priority","Risk"], ascending=[False,False])

    # Build only 1 day per cycle
    df["Build_Qty"] = df["Daily_Demand"]

    df["Hours_Needed"] = df["Build_Qty"] / df["Rate"]

    # =========================
    # PICK BEST MACHINE PER PART
    # =========================

    best_machine = df.loc[df.groupby("Part")["Hours_Needed"].idxmin()]
    df = best_machine.copy()

    # =========================
    # MACHINE-WISE ALLOCATION
    # =========================

    final_plan = []

    for machine, group in df.groupby("Machine"):

        remaining_hours = MACHINE_CAPACITY

        for _, row in group.iterrows():

            if remaining_hours <= 0:
                break

            hours = row["Hours_Needed"]

            if hours < MIN_RUN_HOURS:
                hours = MIN_RUN_HOURS

            if hours > remaining_hours:
                continue

            qty = hours * row["Rate"]

            final_plan.append({
                "Part": row["Part"],
                "Category": row["Category"],
                "Machine": machine,
                "Run_Hours": hours,
                "Production_Qty": qty
            })

            remaining_hours -= hours

    final_df = pd.DataFrame(final_plan)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df, machine_summary

# =========================
# RUN APS
# =========================

hz_long = expand_machines(hz)
vt_long = expand_machines(vt)

hz_plan, hz_machine = run_smart_aps(hz_long)
vt_plan, vt_machine = run_smart_aps(vt_long)

# =========================
# SAVE OUTPUT
# =========================

with pd.ExcelWriter("APS_Smart_Plan.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

print("SMART APS Plan Generated")

In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22
MIN_RUN_HOURS = 4

file_path = "C:/Users/Ex0164/Book1.xlsx"

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")

# =========================
# EXPAND MACHINES
# =========================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Category","Cycle time","Cavity"]
    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]
        df_list.append(temp)

    df_long = pd.concat(df_list, ignore_index=True)

    return df_long

# =========================
# CLEAN DATA
# =========================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Indent"] = df["Indent"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Machine"] != ""]

    df = df[df["Cycle time"].notna()]
    df = df[df["Indent"] > 0]

    return df

# =========================
# SMART APS ENGINE
# =========================

def run_smart_aps(df):

    df["Rate"] = (3600 / df["Cycle time"].replace(0, np.nan)) * df["Cavity"]
    df = df[df["Rate"].notna()]

    df["Daily_Demand"] = df["Indent"] / WORKING_DAYS
    df["Safety"] = df["Daily_Demand"] * 3

    df["Coverage"] = df["Inventory"] / df["Daily_Demand"]
    df["Risk"] = 3 - df["Coverage"]

    df = df[df["Risk"] > 0].copy()

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
    df["Priority"] = df["Category"].map(priority_map)

    df = df.sort_values(by=["Priority","Risk"], ascending=[False,False])

    df["Build_Qty"] = df["Daily_Demand"]
    df["Hours_Needed"] = df["Build_Qty"] / df["Rate"]

    # =========================
    # PICK BEST MACHINE PER PART
    # =========================

    best_machine = df.loc[df.groupby("Part")["Hours_Needed"].idxmin()]
    df = best_machine.copy()

    # =========================
    # MACHINE-WISE ALLOCATION
    # =========================

    final_plan = []

    for machine, group in df.groupby("Machine"):

        remaining_hours = MACHINE_CAPACITY

        for _, row in group.iterrows():

            if remaining_hours <= 0:
                break

            hours = row["Hours_Needed"]

            if hours < MIN_RUN_HOURS:
                hours = MIN_RUN_HOURS

            if hours > remaining_hours:
                continue

            qty = hours * row["Rate"]

            final_plan.append({
                "Part": row["Part"],
                "Category": row["Category"],
                "Machine": machine,
                "Run_Hours": round(hours,2),
                "Production_Qty": round(qty,0)
            })

            remaining_hours -= hours

    final_df = pd.DataFrame(final_plan)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df, machine_summary

# =========================
# PREPARE DATA
# =========================

hz_long = clean_data(expand_machines(hz))
vt_long = clean_data(expand_machines(vt))

# =========================
# RUN APS
# =========================

hz_plan, hz_machine = run_smart_aps(hz_long)
vt_plan, vt_machine = run_smart_aps(vt_long)

# =========================
# SAVE OUTPUT
# =========================

with pd.ExcelWriter("APS_Smart_Plan.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

print("SMART APS Plan Generated Successfully")

In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22
MIN_RUN_HOURS = 4

file_path = "C:/Users/Ex0164/Book1.xlsx"

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")

# =====================================================
# EXPAND MULTIPLE MACHINE COLUMNS
# =====================================================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Category","Cycle time","Cavity"]
    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]
        df_list.append(temp)

    df_long = pd.concat(df_list, ignore_index=True)

    return df_long

# =====================================================
# CLEAN DATA
# =====================================================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Indent"] = df["Indent"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Machine"] != ""]
    df = df[df["Cycle time"].notna()]
    df = df[df["Indent"] > 0]

    return df

# =====================================================
# SMART APS ENGINE
# =====================================================

def run_smart_aps(df):

    # --------------------------
    # Rate calculation
    # --------------------------
    df["Rate"] = (3600 / df["Cycle time"].replace(0, np.nan)) * df["Cavity"]
    df = df[df["Rate"].notna()]

    # --------------------------
    # Demand & Safety
    # --------------------------
    df["Daily_Demand"] = df["Indent"] / WORKING_DAYS
    df["Safety"] = df["Daily_Demand"] * 3

    df["Coverage"] = df["Inventory"] / df["Daily_Demand"]
    df["Risk"] = 3 - df["Coverage"]

    df = df[df["Risk"] > 0].copy()

    # --------------------------
    # Priority Logic
    # --------------------------
    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
    df["Priority"] = df["Category"].map(priority_map)

    df = df.sort_values(by=["Priority","Risk"], ascending=[False,False])

    # --------------------------
    # Build only 1 day at a time
    # --------------------------
    df["Build_Qty"] = df["Daily_Demand"]
    df["Hours_Needed"] = df["Build_Qty"] / df["Rate"]

    # --------------------------
    # Pick BEST machine per part
    # --------------------------
    best_machine = df.loc[df.groupby("Part")["Hours_Needed"].idxmin()]
    df = best_machine.copy()

    # =====================================================
    # MACHINE-WISE SMART ALLOCATION
    # =====================================================

    final_plan = []

    for machine, group in df.groupby("Machine"):

        remaining_hours = MACHINE_CAPACITY
        machine_parts = []

        for _, row in group.iterrows():

            if remaining_hours <= 0:
                break

            hours = row["Hours_Needed"]

            if hours < MIN_RUN_HOURS:
                hours = MIN_RUN_HOURS

            if hours > remaining_hours:
                continue

            machine_parts.append({
                "Part": row["Part"],
                "Category": row["Category"],
                "Rate": row["Rate"],
                "Run_Hours": hours
            })

            remaining_hours -= hours

        # --------------------------
        # SMART UTILISATION
        # --------------------------

        if len(machine_parts) == 1:
            # Only one part → full 22 hrs
            machine_parts[0]["Run_Hours"] = MACHINE_CAPACITY
            remaining_hours = 0

        elif remaining_hours > 0 and len(machine_parts) > 0:

            if remaining_hours <= 2:
                # Small leftover → distribute equally
                extra = remaining_hours / len(machine_parts)
                for part in machine_parts:
                    part["Run_Hours"] += extra
                remaining_hours = 0

            else:
                # Larger leftover → extend runs
                for part in machine_parts:
                    if remaining_hours <= 0:
                        break
                    add = min(remaining_hours, 2)
                    part["Run_Hours"] += add
                    remaining_hours -= add

        # Save results
        for part in machine_parts:
            final_plan.append({
                "Part": part["Part"],
                "Category": part["Category"],
                "Machine": machine,
                "Run_Hours": round(part["Run_Hours"],2),
                "Production_Qty": round(part["Run_Hours"] * part["Rate"],0)
            })

    final_df = pd.DataFrame(final_plan)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df, machine_summary

# =====================================================
# PREPARE DATA
# =====================================================

hz_long = clean_data(expand_machines(hz))
vt_long = clean_data(expand_machines(vt))

# =====================================================
# RUN APS
# =====================================================

hz_plan, hz_machine = run_smart_aps(hz_long)
vt_plan, vt_machine = run_smart_aps(vt_long)

# =====================================================
# SAVE OUTPUT
# =====================================================

with pd.ExcelWriter("APS_Smart_Plan.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

print("FINAL SMART APS PLAN GENERATED SUCCESSFULLY")

In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22
MIN_RUN_HOURS = 4

file_path = "C:/Users/Ex0164/Book1.xlsx"

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")

# =====================================================
# EXPAND MULTIPLE MACHINE COLUMNS
# =====================================================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Daily","Category","Cycle time","Cavity"]
    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]
        df_list.append(temp)

    df_long = pd.concat(df_list, ignore_index=True)

    return df_long

# =====================================================
# CLEAN DATA
# =====================================================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Indent"] = df["Indent"].fillna(0)
    df["Daily"] = df["Daily"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Machine"] != ""]
    df = df[df["Cycle time"].notna()]
    df = df[df["Daily"] > 0]

    return df

# =====================================================
# SMART APS ENGINE
# =====================================================

def run_smart_aps(df):

    # --------------------------
    # Rate calculation
    # --------------------------
    df["Rate"] = (3600 / df["Cycle time"].replace(0, np.nan)) * df["Cavity"]
    df = df[df["Rate"].notna()]

    # --------------------------
    # Safety and Risk
    # --------------------------
    df["Safety"] = df["Daily"] * 3
    df["Coverage"] = df["Inventory"] / df["Daily"]
    df["Risk"] = 3 - df["Coverage"]

    df = df[df["Risk"] > 0].copy()

    # --------------------------
    # Priority Logic
    # --------------------------
    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
    df["Priority"] = df["Category"].map(priority_map)

    df = df.sort_values(by=["Priority","Risk"], ascending=[False,False])

    # --------------------------
    # Build only 1 day at a time
    # --------------------------
    df["Build_Qty"] = df["Daily"]
    df["Hours_Needed"] = df["Build_Qty"] / df["Rate"]

    # --------------------------
    # Pick BEST machine per part
    # --------------------------
    best_machine = df.loc[df.groupby("Part")["Hours_Needed"].idxmin()]
    df = best_machine.copy()

    # =====================================================
    # MACHINE-WISE SMART ALLOCATION
    # =====================================================

    final_plan = []

    for machine, group in df.groupby("Machine"):

        remaining_hours = MACHINE_CAPACITY
        machine_parts = []

        for _, row in group.iterrows():

            if remaining_hours <= 0:
                break

            hours = row["Hours_Needed"]

            if hours < MIN_RUN_HOURS:
                hours = MIN_RUN_HOURS

            if hours > remaining_hours:
                continue

            machine_parts.append({
                "Part": row["Part"],
                "Category": row["Category"],
                "Inventory": row["Inventory"],
                "Indent": row["Indent"],
                "Daily": row["Daily"],
                "Cycle time": row["Cycle time"],
                "Cavity": row["Cavity"],
                "Rate": row["Rate"],
                "Run_Hours": hours
            })

            remaining_hours -= hours

        # --------------------------
        # SMART UTILISATION
        # --------------------------

        if len(machine_parts) == 1:
            machine_parts[0]["Run_Hours"] = MACHINE_CAPACITY
            remaining_hours = 0

        elif remaining_hours > 0 and len(machine_parts) > 0:

            if remaining_hours <= 2:
                extra = remaining_hours / len(machine_parts)
                for part in machine_parts:
                    part["Run_Hours"] += extra
                remaining_hours = 0

            else:
                for part in machine_parts:
                    if remaining_hours <= 0:
                        break
                    add = min(remaining_hours, 2)
                    part["Run_Hours"] += add
                    remaining_hours -= add

        # Save results
        for part in machine_parts:
            final_plan.append({
                "Part": part["Part"],
                "Category": part["Category"],
                "Inventory": part["Inventory"],
                "Indent": part["Indent"],
                "Daily": part["Daily"],
                "Cycle time": part["Cycle time"],
                "Cavity": part["Cavity"],
                "Rate": round(part["Rate"],2),
                "Machine": machine,
                "Run_Hours": round(part["Run_Hours"],2),
                "Production_Qty": round(part["Run_Hours"] * part["Rate"],0)
            })

    final_df = pd.DataFrame(final_plan)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df, machine_summary

# =====================================================
# PREPARE DATA
# =====================================================

hz_long = clean_data(expand_machines(hz))
vt_long = clean_data(expand_machines(vt))

# =====================================================
# RUN APS
# =====================================================

hz_plan, hz_machine = run_smart_aps(hz_long)
vt_plan, vt_machine = run_smart_aps(vt_long)

# =====================================================
# SAVE OUTPUT
# =====================================================

with pd.ExcelWriter("APS_Smart_Plan.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

print("FINAL SMART APS PLAN GENERATED SUCCESSFULLY")

In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22
MIN_RUN_HOURS = 4
STRUCTURAL_THRESHOLD = 1.2  # 20% margin rule

file_path = "C:/Users/Ex0164/Book1.xlsx"

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")

# =====================================================
# EXPAND MACHINE COLUMNS
# =====================================================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Daily","Category","Cycle time","Cavity"]
    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]
        df_list.append(temp)

    return pd.concat(df_list, ignore_index=True)

# =====================================================
# CLEAN DATA
# =====================================================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Daily"] = df["Daily"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Machine"] != ""]
    df = df[df["Cycle time"].notna()]
    df = df[df["Daily"] > 0]

    return df

# =====================================================
# SMART APS WITH STRUCTURAL RUNNER LOGIC
# =====================================================

def run_smart_aps(df):

    df["Rate"] = (3600 / df["Cycle time"].replace(0, np.nan)) * df["Cavity"]
    df = df[df["Rate"].notna()]

    df["Safety"] = df["Daily"] * 3
    df["Coverage"] = df["Inventory"] / df["Daily"]
    df["Deficit"] = df["Safety"] - df["Inventory"]

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
    df["Priority"] = df["Category"].map(priority_map)

    df = df.sort_values(by=["Priority","Deficit"], ascending=[False,False])

    final_plan = []

    for machine, group in df.groupby("Machine"):

        remaining_hours = MACHINE_CAPACITY

        for _, row in group.iterrows():

            if remaining_hours <= 0:
                break

            rate = row["Rate"]
            daily = row["Daily"]

            max_possible_output = rate * remaining_hours
            net_growth = max_possible_output - daily

            # ================================
            # STRUCTURAL RUNNER DETECTION
            # ================================

            if rate * remaining_hours <= STRUCTURAL_THRESHOLD * daily:

                # Accept frequent scheduling
                hours = max(daily / rate, MIN_RUN_HOURS)
                hours = min(hours, remaining_hours)

            else:
                # Try to build toward safety
                hours_needed_for_safety = max(row["Deficit"] / rate, 0)
                hours = max(hours_needed_for_safety, MIN_RUN_HOURS)
                hours = min(hours, remaining_hours)

            final_plan.append({
                "Part": row["Part"],
                "Category": row["Category"],
                "Inventory": row["Inventory"],
                "Indent": row["Indent"],
                "Daily": row["Daily"],
                "Cycle time": row["Cycle time"],
                "Cavity": row["Cavity"],
                "Rate": round(rate,2),
                "Machine": machine,
                "Run_Hours": round(hours,2),
                "Production_Qty": round(hours * rate,0)
            })

            remaining_hours -= hours

    final_df = pd.DataFrame(final_plan)
    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df, machine_summary

# =====================================================
# RUN SYSTEM
# =====================================================

hz_long = clean_data(expand_machines(hz))
vt_long = clean_data(expand_machines(vt))

hz_plan, hz_machine = run_smart_aps(hz_long)
vt_plan, vt_machine = run_smart_aps(vt_long)

with pd.ExcelWriter("APS_Structural_Smart_Plan.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

print("STRUCTURAL-AWARE SMART APS GENERATED SUCCESSFULLY")

In [ ]:
import pandas as pd
import numpy as np
import logging
from sklearn.linear_model import LinearRegression

# ============================
# CONFIGURATION
# ============================

CONFIG = {
    "SIMULATIONS": 1000,
    "SHORTAGE_PENALTY": 10,
    "HOLDING_PENALTY": 1,
    "AVAILABLE_HOURS": 22,
    "SETUP_HOURS": 40/60,
    "MIN_RUN_HOURS": 4,
    "MOVE_THRESHOLD": 2
}

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx"
memory_path = "C:/Users/Ex0164/machine_memory.xlsx"

logging.basicConfig(level=logging.INFO)

# ============================
# LOAD DATA
# ============================

hz = pd.read_excel(file_path,"HZ")
vt = pd.read_excel(file_path,"VT")
daily = pd.read_excel(daily_path)
memory = pd.read_excel(memory_path)

# ============================
# DEMAND ADAPTIVE STATS
# ============================

def compute_adaptive_stats(daily_df):

    stats_list = []

    for part,group in daily_df.groupby("Part"):

        demand = group["Demand"].dropna()

        if len(demand)<3:
            continue

        mean = demand.mean()
        std = demand.std()

        X = np.arange(len(demand)).reshape(-1,1)
        y = demand.values

        model = LinearRegression()
        model.fit(X,y)

        slope = model.coef_[0]

        future_mean = mean + slope

        stats_list.append({
            "Part":part,
            "Mean_Actual":future_mean,
            "Std_Deviation":std
        })

    return pd.DataFrame(stats_list)

stats = compute_adaptive_stats(daily)

# ============================
# EXPAND MACHINES
# ============================

def expand_machines(df):

    base_cols = [
        "Part","Inventory","Indent",
        "Daily","Category","Cycle time","Cavity"
    ]

    machine_cols = [c for c in df.columns if c.startswith("Machine")]

    frames = []

    for m in machine_cols:

        temp = df[base_cols].copy()
        temp["Machine"] = df[m]
        frames.append(temp)

    return pd.concat(frames)

# ============================
# CLEAN DATA
# ============================

def clean_data(df):

    df["Inventory"]=df["Inventory"].fillna(0)
    df["Indent"]=df["Indent"].fillna(0)
    df["Daily"]=df["Daily"].fillna(0)
    df["Cavity"]=df["Cavity"].fillna(1)

    df=df[df["Machine"].notna()]
    df=df[df["Cycle time"].notna()]
    df=df[df["Daily"]>0]

    return df

hz_long = clean_data(expand_machines(hz)).merge(stats,on="Part",how="left")
vt_long = clean_data(expand_machines(vt)).merge(stats,on="Part",how="left")

# ============================
# PRODUCTION RATE
# ============================

def compute_rate(df):

    df["Rate"] = (3600/df["Cycle time"]) * df["Cavity"]

    return df[df["Rate"].notna()]

# ============================
# OPTIMIZER
# ============================

def optimize_part(row):

    inventory = row["Inventory"]
    demand = row["Daily"]

    mean = row["Mean_Actual"]
    std = row["Std_Deviation"]

    if pd.isna(mean):
        mean=demand

    if pd.isna(std):
        std=demand*0.2

    rate=row["Rate"]

    max_qty=int(row["Indent"]*1.2)

    qty_options=np.arange(0,max_qty+1,50)

    simulated=np.random.normal(mean,std,CONFIG["SIMULATIONS"])

    future_stock=inventory+qty_options[:,None]-demand

    shortage=np.maximum(0,simulated-future_stock)

    expected_shortage=shortage.mean(axis=1)

    expected_inventory=np.maximum(
        0,
        future_stock.mean(axis=1)-simulated.mean()
    )

    setup=np.where(qty_options>0,rate*CONFIG["SETUP_HOURS"],0)

    cost=(CONFIG["SHORTAGE_PENALTY"]*expected_shortage+
          CONFIG["HOLDING_PENALTY"]*expected_inventory+
          setup)

    hours=qty_options/rate

    feasible=hours<=CONFIG["AVAILABLE_HOURS"]

    if feasible.any():

        idx=np.argmin(np.where(feasible,cost,np.inf))

        return qty_options[idx],cost[idx]

    return 0,np.inf

# ============================
# MACHINE MEMORY
# ============================

memory_dict = dict(zip(memory["Machine"],memory["Last_Part"]))

# ============================
# APS SCHEDULING
# ============================

def run_dynamic_aps(df):

    df = compute_rate(df)

    priority_map={"Runner":3,"Repeater":2,"Stranger":1}

    df["Priority"]=df["Category"].map(priority_map)

    plan=[]

    for machine,group in df.groupby("Machine"):

        remaining=CONFIG["AVAILABLE_HOURS"]

        last_part=memory_dict.get(machine,None)

        # continue last part first
        if last_part:

            part_rows=group[group["Part"]==last_part]

            for _,row in part_rows.iterrows():

                qty,cost=optimize_part(row)

                hours=qty/row["Rate"]

                hours=max(hours,CONFIG["MIN_RUN_HOURS"])

                if hours<=remaining:

                    plan.append({
                        "Part":row["Part"],
                        "Machine":machine,
                        "Run_Hours":hours,
                        "Production_Qty":hours*row["Rate"],
                        "Priority":row["Priority"]
                    })

                    remaining-=hours

        # remaining parts
        for _,row in group.iterrows():

            if remaining<=0:
                break

            if row["Part"]==last_part:
                continue

            qty,cost=optimize_part(row)

            hours=qty/row["Rate"]

            hours=max(hours,CONFIG["MIN_RUN_HOURS"])

            if hours<=remaining:

                plan.append({
                    "Part":row["Part"],
                    "Machine":machine,
                    "Run_Hours":hours,
                    "Production_Qty":hours*row["Rate"],
                    "Priority":row["Priority"]
                })

                remaining-=hours

    return pd.DataFrame(plan)

# ============================
# MACHINE LOAD BALANCING
# ============================

def balance_machines(plan):

    machine_hours = plan.groupby("Machine")["Run_Hours"].sum()

    for machine in machine_hours.index:

        used=machine_hours[machine]

        if used>=CONFIG["AVAILABLE_HOURS"]:
            continue

        free=CONFIG["AVAILABLE_HOURS"]-used

        if free<CONFIG["MOVE_THRESHOLD"]:
            continue

        for idx,row in plan.iterrows():

            src_machine=row["Machine"]

            if src_machine==machine:
                continue

            src_hours=machine_hours[src_machine]

            if src_hours<=CONFIG["AVAILABLE_HOURS"]:
                continue

            move_hours=min(row["Run_Hours"],free)

            plan.loc[idx,"Machine"]=machine

            machine_hours[src_machine]-=move_hours
            machine_hours[machine]+=move_hours

            break

    return plan

# ============================
# RUN APS
# ============================

hz_plan = run_dynamic_aps(hz_long)
vt_plan = run_dynamic_aps(vt_long)

hz_plan = balance_machines(hz_plan)
vt_plan = balance_machines(vt_plan)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Final_Planner.xlsx") as writer:

    hz_plan.to_excel(writer,"HZ_Plan",index=False)
    vt_plan.to_excel(writer,"VT_Plan",index=False)

logging.info("APS Planner with load balancing + memory completed")

In [ ]:
import pandas as pd
import numpy as np
import logging
from sklearn.linear_model import LinearRegression

logging.basicConfig(level=logging.INFO)

# ============================
# CONFIGURATION
# ============================

CONFIG = {
    "SIMULATIONS":1000,
    "AVAILABLE_HOURS":22,
    "SHORTAGE_PENALTY":10,
    "HOLDING_PENALTY":1,
    "SETUP_HOURS":40/60,
    "MIN_RUN_HOURS":4,
    "MOVE_THRESHOLD":2
}

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx"

# ============================
# LOAD DATA
# ============================

hz = pd.read_excel(file_path,"HZ")
vt = pd.read_excel(file_path,"VT")
daily = pd.read_excel(daily_path)

# ============================
# DEMAND STATISTICS
# ============================

def compute_adaptive_stats(daily_df):

    stats = []

    for part,group in daily_df.groupby("Part"):

        demand = group["Demand"].dropna()

        if len(demand)<3:
            continue

        mean = demand.mean()
        std = demand.std()

        X = np.arange(len(demand)).reshape(-1,1)
        y = demand.values

        model = LinearRegression()
        model.fit(X,y)

        slope = model.coef_[0]

        future_mean = mean + slope

        stats.append({
            "Part":part,
            "Mean_Actual":future_mean,
            "Std_Deviation":std
        })

    return pd.DataFrame(stats)

stats = compute_adaptive_stats(daily)

# ============================
# EXPAND MACHINE COLUMNS
# ============================

def expand_machines(df):

    base_cols=[
        "Part","Inventory","Indent",
        "Daily","Category","Cycle time","Cavity"
    ]

    machine_cols=[c for c in df.columns if c.startswith("Machine")]

    frames=[]

    for m in machine_cols:

        temp=df[base_cols].copy()
        temp["Machine"]=df[m]

        frames.append(temp)

    return pd.concat(frames)

# ============================
# CLEAN DATA
# ============================

def clean_data(df):

    df["Inventory"]=df["Inventory"].fillna(0)
    df["Indent"]=df["Indent"].fillna(0)
    df["Daily"]=df["Daily"].fillna(0)
    df["Cavity"]=df["Cavity"].fillna(1)

    df=df[df["Machine"].notna()]
    df=df[df["Cycle time"].notna()]
    df=df[df["Daily"]>0]

    return df

hz_long = clean_data(expand_machines(hz)).merge(stats,on="Part",how="left")
vt_long = clean_data(expand_machines(vt)).merge(stats,on="Part",how="left")

# ============================
# PRODUCTION RATE
# ============================

def compute_rate(df):

    df["Rate"] = (3600/df["Cycle time"]) * df["Cavity"]

    return df[df["Rate"].notna()]

# ============================
# MONTE CARLO OPTIMIZER
# ============================

def optimize_part(row):

    inventory=row["Inventory"]
    demand=row["Daily"]

    mean=row["Mean_Actual"]
    std=row["Std_Deviation"]

    if pd.isna(mean):
        mean=demand

    if pd.isna(std):
        std=demand*0.2

    rate=row["Rate"]

    max_qty=int(row["Indent"]*1.2)

    qty_options=np.arange(0,max_qty+1,50)

    simulated=np.random.normal(mean,std,CONFIG["SIMULATIONS"])

    future_stock=inventory+qty_options[:,None]-demand

    shortage=np.maximum(0,simulated-future_stock)

    expected_shortage=shortage.mean(axis=1)

    expected_inventory=np.maximum(
        0,
        future_stock.mean(axis=1)-simulated.mean()
    )

    setup=np.where(qty_options>0,rate*CONFIG["SETUP_HOURS"],0)

    cost=(CONFIG["SHORTAGE_PENALTY"]*expected_shortage+
          CONFIG["HOLDING_PENALTY"]*expected_inventory+
          setup)

    hours=qty_options/rate

    feasible=hours<=CONFIG["AVAILABLE_HOURS"]

    if feasible.any():

        idx=np.argmin(np.where(feasible,cost,np.inf))

        return qty_options[idx],cost[idx]

    return 0,np.inf

# ============================
# SEQUENCING SCORE
# ============================

def compute_sequence_score(df):

    df["Urgency"]=df["Daily"]/(df["Inventory"]+1)

    df["TimeToManufacture"]=df["Indent"]/df["Rate"]

    priority_map={"Runner":3,"Repeater":2,"Stranger":1}

    df["Priority"]=df["Category"].map(priority_map)

    df["Score"]=(
        3*df["Priority"] +
        2*df["Urgency"] +
        1.5*df["Daily"] +
        1/(df["Cycle time"]+1) +
        0.5*df["TimeToManufacture"]
    )

    return df

# ============================
# APS SCHEDULER
# ============================

def run_dynamic_aps(df):

    df=compute_rate(df)

    df=compute_sequence_score(df)

    plan=[]

    for machine,group in df.groupby("Machine"):

        remaining=CONFIG["AVAILABLE_HOURS"]

        group=group.sort_values("Score",ascending=False)

        for _,row in group.iterrows():

            if remaining<=0:
                break

            qty,cost=optimize_part(row)

            hours=qty/row["Rate"]

            hours=max(hours,CONFIG["MIN_RUN_HOURS"])

            if hours<=remaining:

                plan.append({
                    "Machine":machine,
                    "Part":row["Part"],
                    "Run_Hours":hours,
                    "Production_Qty":hours*row["Rate"],
                    "Score":row["Score"]
                })

                remaining-=hours

    return pd.DataFrame(plan)

# ============================
# MACHINE LOAD BALANCING
# ============================

def balance_machines(plan):

    machine_hours=plan.groupby("Machine")["Run_Hours"].sum()

    for machine in machine_hours.index:

        used=machine_hours[machine]

        free=CONFIG["AVAILABLE_HOURS"]-used

        if free<CONFIG["MOVE_THRESHOLD"]:
            continue

        for idx,row in plan.iterrows():

            src=row["Machine"]

            if src==machine:
                continue

            if machine_hours[src]<=CONFIG["AVAILABLE_HOURS"]:
                continue

            move=min(row["Run_Hours"],free)

            plan.loc[idx,"Machine"]=machine

            machine_hours[src]-=move
            machine_hours[machine]+=move

            break

    return plan

# ============================
# RUN APS
# ============================

hz_plan=run_dynamic_aps(hz_long)
vt_plan=run_dynamic_aps(vt_long)

hz_plan=balance_machines(hz_plan)
vt_plan=balance_machines(vt_plan)

# ============================
# ASSIGN SEQUENCE
# ============================

hz_plan["Sequence"]=hz_plan.groupby("Machine").cumcount()+1
vt_plan["Sequence"]=vt_plan.groupby("Machine").cumcount()+1

# ============================
# MACHINE MEMORY (FOR TOMORROW)
# ============================

machine_memory=hz_plan.sort_values("Sequence").groupby("Machine").last()["Part"]

memory_df=machine_memory.reset_index()
memory_df.columns=["Machine","Last_Part"]

memory_df.to_excel("machine_memory_next_day.xlsx",index=False)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Final_Planner.xlsx") as writer:

    hz_plan.to_excel(writer,"HZ_Plan",index=False)
    vt_plan.to_excel(writer,"VT_Plan",index=False)

logging.info("APS planner completed successfully")

In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# CONFIGURATION
# ============================

CONFIG = {
    "SIMULATIONS": 1000,
    "SHORTAGE_PENALTY": 10,
    "HOLDING_PENALTY": 1,
    "AVAILABLE_HOURS": 22,
    "SETUP_HOURS": 40 / 60,
    "MIN_RUN_HOURS": 4,
    "MOVE_THRESHOLD": 2
}

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

# ============================
# LOGGING
# ============================

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================
# LOAD FILES
# ============================

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")
stats = pd.read_excel(file_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix")

hz_matrix = hz_matrix.set_index("Part")
vt_matrix = vt_matrix.set_index("Part")

# ============================
# EXPAND MACHINES
# ============================

def expand_machines(df):

    base_cols = [
        "Part","Inventory","Indent",
        "Daily","Category","Cycle time","Cavity"
    ]

    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:

        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]

        df_list.append(temp)

    return pd.concat(df_list, ignore_index=True)

# ============================
# CLEAN DATA
# ============================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Indent"] = df["Indent"].fillna(0)
    df["Daily"] = df["Daily"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Cycle time"].notna()]
    df = df[df["Daily"] > 0]

    return df

hz_long = clean_data(expand_machines(hz)).merge(stats, on="Part", how="left")
vt_long = clean_data(expand_machines(vt)).merge(stats, on="Part", how="left")

# ============================
# SIMULATION OPTIMIZER
# ============================

def optimize_part(row):

    inventory = row["Inventory"]
    demand_tomorrow = row["Daily"]
    mean = row["Mean_Actual"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    max_qty = int(row["Indent"] * 1.2)

    qty_options = np.arange(0, max_qty + 1, 50)

    simulated_demand = np.random.normal(mean, std, CONFIG["SIMULATIONS"])

    future_stock = inventory + qty_options[:, None] - demand_tomorrow

    shortage = np.maximum(0, simulated_demand - future_stock)

    expected_shortage = shortage.mean(axis=1)

    expected_inventory = np.maximum(
        0,
        future_stock.mean(axis=1) - simulated_demand.mean()
    )

    setup_penalty = np.where(
        qty_options > 0,
        rate * CONFIG["SETUP_HOURS"],
        0
    )

    cost = (
        CONFIG["SHORTAGE_PENALTY"] * expected_shortage +
        CONFIG["HOLDING_PENALTY"] * expected_inventory +
        setup_penalty
    )

    hours_needed = qty_options / rate

    feasible = hours_needed <= CONFIG["AVAILABLE_HOURS"]

    if feasible.any():

        best_idx = np.argmin(np.where(feasible, cost, np.inf))

        return qty_options[best_idx], cost[best_idx]

    return 0, np.inf

# ============================
# MACHINE BALANCING USING MATRIX
# ============================

def balance_using_matrix(plan_df, matrix):

    machine_hours = plan_df.groupby("Machine")["Run_Hours"].sum().to_dict()

    for idx, row in plan_df.iterrows():

        part = row["Part"]
        current_machine = row["Machine"]
        run_hours = row["Run_Hours"]

        if part not in matrix.index:
            continue

        if machine_hours[current_machine] <= CONFIG["AVAILABLE_HOURS"]:
            continue

        compatible = matrix.loc[part]

        compatible_machines = compatible[
            compatible == 1
        ].index.tolist()

        for m in compatible_machines:

            if m == current_machine:
                continue

            if m not in machine_hours:
                machine_hours[m] = 0

            free = CONFIG["AVAILABLE_HOURS"] - machine_hours[m]

            if free > CONFIG["MOVE_THRESHOLD"]:

                plan_df.loc[idx, "Machine"] = m

                machine_hours[current_machine] -= run_hours
                machine_hours[m] += run_hours

                break

    return plan_df

# ============================
# SEQUENCING LOGIC
# ============================

def sequence_parts(group):

    group["Time_To_Manufacture"] = group["Indent"] / group["Rate"]

    group["Sequence_Score"] = (
        group["Priority"] * 3
        + (1 / (group["Inventory"] + 1)) * 2
        + (group["Daily"]) * 1.5
        + (1 / (group["Cycle time"] + 1))
        + group["Time_To_Manufacture"] * 0.5
    )

    group = group.sort_values("Sequence_Score", ascending=False)

    group["Sequence"] = range(1, len(group) + 1)

    return group

# ============================
# RUN APS
# ============================

def run_dynamic_aps(df, matrix):

    df["Rate"] = (3600 / df["Cycle time"]) * df["Cavity"]

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}

    df["Priority"] = df["Category"].map(priority_map)

    final_plan = []

    for machine, group in df.groupby("Machine"):

        remaining_hours = CONFIG["AVAILABLE_HOURS"]

        for _, row in group.iterrows():

            if remaining_hours <= 0:
                break

            qty, cost = optimize_part(row)

            hours = qty / row["Rate"]

            if hours > 0:

                hours = max(hours, CONFIG["MIN_RUN_HOURS"])

                if hours <= remaining_hours:

                    final_plan.append({
                        "Part":row["Part"],
                        "Machine":machine,
                        "Run_Hours":hours,
                        "Production_Qty":hours * row["Rate"],
                        "Priority":row["Priority"],
                        "Inventory":row["Inventory"],
                        "Daily":row["Daily"],
                        "Cycle time":row["Cycle time"],
                        "Indent":row["Indent"],
                        "Rate":row["Rate"]
                    })

                    remaining_hours -= hours

    final_df = pd.DataFrame(final_plan)

    final_df = balance_using_matrix(final_df, matrix)

    final_df = final_df.groupby("Machine", group_keys=False).apply(sequence_parts)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df, machine_summary

# ============================
# RUN APS
# ============================

hz_plan, hz_machine = run_dynamic_aps(hz_long, hz_matrix)
vt_plan, vt_machine = run_dynamic_aps(vt_long, vt_matrix)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Dynamic_Hybrid_Plan_Simulation.xlsx", engine="xlsxwriter") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

logging.info("APS with compatibility matrix completed")

In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# CONFIGURATION
# ============================

CONFIG = {
    "SIMULATIONS": 1000,
    "SHORTAGE_PENALTY": 10,
    "HOLDING_PENALTY": 1,
    "AVAILABLE_HOURS": 22,
    "SETUP_HOURS": 40 / 60,
    "MIN_RUN_HOURS": 4,
    "MOVE_THRESHOLD": 2
}

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

# ============================
# LOGGING
# ============================

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================
# LOAD FILES
# ============================

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")
stats = pd.read_excel(file_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix")

hz_matrix = hz_matrix.set_index("Part")
vt_matrix = vt_matrix.set_index("Part")

# ============================
# EXPAND MACHINES
# ============================

def expand_machines(df):

    base_cols = [
        "Part","Inventory","Indent",
        "Daily","Category","Cycle time","Cavity"
    ]

    machine_cols = [c for c in df.columns if c.startswith("Machine")]

    frames = []

    for m in machine_cols:

        temp = df[base_cols].copy()
        temp["Machine"] = df[m]

        frames.append(temp)

    return pd.concat(frames, ignore_index=True)

# ============================
# CLEAN DATA
# ============================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Indent"] = df["Indent"].fillna(0)
    df["Daily"] = df["Daily"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Cycle time"].notna()]
    df = df[df["Daily"] > 0]

    return df

hz_long = clean_data(expand_machines(hz)).merge(stats, on="Part", how="left")
vt_long = clean_data(expand_machines(vt)).merge(stats, on="Part", how="left")

# ============================
# MONTE CARLO OPTIMIZER
# ============================

def optimize_part(row):

    inventory = row["Inventory"]
    demand = row["Daily"]
    mean = row["Mean_Actual"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    max_qty = int(row["Indent"] * 1.2)

    qty_options = np.arange(0, max_qty + 1, 50)

    simulated = np.random.normal(mean, std, CONFIG["SIMULATIONS"])

    future_stock = inventory + qty_options[:, None] - demand

    shortage = np.maximum(0, simulated - future_stock)

    expected_shortage = shortage.mean(axis=1)

    expected_inventory = np.maximum(
        0,
        future_stock.mean(axis=1) - simulated.mean()
    )

    setup_penalty = np.where(
        qty_options > 0,
        rate * CONFIG["SETUP_HOURS"],
        0
    )

    cost = (
        CONFIG["SHORTAGE_PENALTY"] * expected_shortage +
        CONFIG["HOLDING_PENALTY"] * expected_inventory +
        setup_penalty
    )

    hours_needed = qty_options / rate

    feasible = hours_needed <= CONFIG["AVAILABLE_HOURS"]

    if feasible.any():

        best_idx = np.argmin(np.where(feasible, cost, np.inf))

        return qty_options[best_idx], cost[best_idx]

    return 0, np.inf

# ============================
# MACHINE BALANCING
# ============================

def balance_using_matrix(plan_df, matrix):

    machine_hours = plan_df.groupby("Machine")["Run_Hours"].sum().to_dict()

    for idx, row in plan_df.iterrows():

        part = row["Part"]
        current_machine = row["Machine"]
        hours = row["Run_Hours"]

        if part not in matrix.index:
            continue

        if machine_hours[current_machine] <= CONFIG["AVAILABLE_HOURS"]:
            continue

        compatible = matrix.loc[part]

        machines = compatible[compatible == 1].index.tolist()

        for m in machines:

            if m == current_machine:
                continue

            if m not in machine_hours:
                machine_hours[m] = 0

            free = CONFIG["AVAILABLE_HOURS"] - machine_hours[m]

            if free > CONFIG["MOVE_THRESHOLD"]:

                plan_df.loc[idx, "Machine"] = m

                machine_hours[current_machine] -= hours
                machine_hours[m] += hours

                break

    return plan_df

# ============================
# UTILIZE REMAINING TIME
# ============================

def utilize_remaining_time(plan_df, df, matrix):

    adjusted = []

    for machine, group in plan_df.groupby("Machine"):

        used = group["Run_Hours"].sum()

        remaining = CONFIG["AVAILABLE_HOURS"] - used

        # add another part
        if remaining >= CONFIG["MIN_RUN_HOURS"]:

            possible_parts = df[df["Machine"] == machine]

            for _, row in possible_parts.iterrows():

                part = row["Part"]

                if part in matrix.index and machine in matrix.columns:

                    if matrix.loc[part, machine] != 1:
                        continue

                rate = row["Rate"]

                hours_needed = CONFIG["MIN_RUN_HOURS"]

                if hours_needed <= remaining:

                    new_row = {
                        "Part": part,
                        "Machine": machine,
                        "Run_Hours": hours_needed,
                        "Production_Qty": hours_needed * rate,
                        "Rate": rate,
                        "Priority": row["Priority"],
                        "Inventory": row["Inventory"],
                        "Daily": row["Daily"],
                        "Cycle time": row["Cycle time"],
                        "Indent": row["Indent"]
                    }

                    group = pd.concat([group, pd.DataFrame([new_row])])

                    remaining -= hours_needed

                    break

        # extend existing parts
        elif remaining > 0:

            extra = remaining / len(group)

            group["Run_Hours"] += extra
            group["Production_Qty"] = group["Run_Hours"] * group["Rate"]

        adjusted.append(group)

    return pd.concat(adjusted, ignore_index=True)

# ============================
# SEQUENCING
# ============================

def sequence_parts(group):

    group["Time_To_Manufacture"] = group["Indent"] / group["Rate"]

    group["Sequence_Score"] = (
        group["Priority"] * 3
        + (1/(group["Inventory"]+1))*2
        + group["Daily"]*1.5
        + (1/(group["Cycle time"]+1))
        + group["Time_To_Manufacture"]*0.5
    )

    group = group.sort_values("Sequence_Score", ascending=False)

    group["Sequence"] = range(1, len(group)+1)

    return group

# ============================
# RUN APS
# ============================

def run_dynamic_aps(df, matrix):

    df["Rate"] = (3600 / df["Cycle time"]) * df["Cavity"]

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}

    df["Priority"] = df["Category"].map(priority_map)

    plan = []

    for machine, group in df.groupby("Machine"):

        remaining = CONFIG["AVAILABLE_HOURS"]

        for _, row in group.iterrows():

            if remaining <= 0:
                break

            qty, cost = optimize_part(row)

            hours = qty / row["Rate"]

            if hours > 0:

                hours = max(hours, CONFIG["MIN_RUN_HOURS"])

                if hours <= remaining:

                    plan.append({
                        "Part":row["Part"],
                        "Machine":machine,
                        "Run_Hours":hours,
                        "Production_Qty":hours*row["Rate"],
                        "Rate":row["Rate"],
                        "Priority":row["Priority"],
                        "Inventory":row["Inventory"],
                        "Daily":row["Daily"],
                        "Cycle time":row["Cycle time"],
                        "Indent":row["Indent"]
                    })

                    remaining -= hours

    final_df = pd.DataFrame(plan)

    final_df = balance_using_matrix(final_df, matrix)

    final_df = utilize_remaining_time(final_df, df, matrix)

    final_df = final_df.groupby("Machine", group_keys=False).apply(sequence_parts)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df, machine_summary

# ============================
# RUN APS
# ============================

hz_plan, hz_machine = run_dynamic_aps(hz_long, hz_matrix)
vt_plan, vt_machine = run_dynamic_aps(vt_long, vt_matrix)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Dynamic_Hybrid_Plan_Simulation.xlsx", engine="xlsxwriter") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

logging.info("APS Dynamic Hybrid Planner completed")

In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# CONFIGURATION
# ============================

CONFIG = {
    "SIMULATIONS": 1000,
    "SHORTAGE_PENALTY": 10,
    "HOLDING_PENALTY": 1,
    "AVAILABLE_HOURS": 22,
    "SETUP_HOURS": 40 / 60,
    "MIN_RUN_HOURS": 4,
    "MOVE_THRESHOLD": 2
}

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

# ============================
# LOGGING
# ============================

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================
# LOAD FILES
# ============================

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")
stats = pd.read_excel(file_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix")

hz_matrix = hz_matrix.set_index("Part")
vt_matrix = vt_matrix.set_index("Part")

# ============================
# EXPAND MACHINES
# ============================

def expand_machines(df):

    base_cols = [
        "Part","Inventory","Indent",
        "Daily","Category","Cycle time","Cavity"
    ]

    machine_cols = [c for c in df.columns if c.startswith("Machine")]

    frames = []

    for m in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[m]
        frames.append(temp)

    return pd.concat(frames, ignore_index=True)

# ============================
# CLEAN DATA
# ============================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Indent"] = df["Indent"].fillna(0)
    df["Daily"] = df["Daily"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Cycle time"].notna()]
    df = df[df["Daily"] > 0]

    return df

hz_long = clean_data(expand_machines(hz)).merge(stats, on="Part", how="left")
vt_long = clean_data(expand_machines(vt)).merge(stats, on="Part", how="left")

# ============================
# MONTE CARLO OPTIMIZER
# ============================

def optimize_part(row):

    inventory = row["Inventory"]
    demand = row["Daily"]
    mean = row["Mean_Actual"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    max_qty = int(row["Indent"] * 1.2)

    qty_options = np.arange(0, max_qty + 1, 50)

    simulated = np.random.normal(mean, std, CONFIG["SIMULATIONS"])

    future_stock = inventory + qty_options[:, None] - demand

    shortage = np.maximum(0, simulated - future_stock)

    expected_shortage = shortage.mean(axis=1)

    expected_inventory = np.maximum(
        0,
        future_stock.mean(axis=1) - simulated.mean()
    )

    setup_penalty = np.where(
        qty_options > 0,
        rate * CONFIG["SETUP_HOURS"],
        0
    )

    cost = (
        CONFIG["SHORTAGE_PENALTY"] * expected_shortage +
        CONFIG["HOLDING_PENALTY"] * expected_inventory +
        setup_penalty
    )

    hours_needed = qty_options / rate

    feasible = hours_needed <= CONFIG["AVAILABLE_HOURS"]

    if feasible.any():
        best_idx = np.argmin(np.where(feasible, cost, np.inf))
        return qty_options[best_idx], cost[best_idx]

    return 0, np.inf

# ============================
# MACHINE BALANCING
# ============================

def balance_using_matrix(plan_df, matrix):

    machine_hours = plan_df.groupby("Machine")["Run_Hours"].sum().to_dict()

    for idx,row in plan_df.iterrows():

        part = row["Part"]
        machine = row["Machine"]
        hours = row["Run_Hours"]

        if part not in matrix.index:
            continue

        if machine_hours[machine] <= CONFIG["AVAILABLE_HOURS"]:
            continue

        compatible = matrix.loc[part]
        machines = compatible[compatible == 1].index.tolist()

        for m in machines:

            if m == machine:
                continue

            if m not in machine_hours:
                machine_hours[m] = 0

            free = CONFIG["AVAILABLE_HOURS"] - machine_hours[m]

            if free > CONFIG["MOVE_THRESHOLD"]:

                plan_df.loc[idx,"Machine"] = m

                machine_hours[machine] -= hours
                machine_hours[m] += hours

                break

    return plan_df

# ============================
# UTILIZE REMAINING TIME
# ============================

def utilize_remaining_time(plan_df, df, matrix):

    adjusted = []

    for machine, group in plan_df.groupby("Machine"):

        used = group["Run_Hours"].sum()
        remaining = CONFIG["AVAILABLE_HOURS"] - used

        if remaining >= CONFIG["MIN_RUN_HOURS"]:

            possible_parts = df[df["Machine"] == machine]

            for _, row in possible_parts.iterrows():

                part = row["Part"]

                if part in matrix.index and machine in matrix.columns:
                    if matrix.loc[part, machine] != 1:
                        continue

                rate = row["Rate"]

                if CONFIG["MIN_RUN_HOURS"] <= remaining:

                    new_row = {
                        "Part":part,
                        "Machine":machine,
                        "Run_Hours":CONFIG["MIN_RUN_HOURS"],
                        "Production_Qty":CONFIG["MIN_RUN_HOURS"]*rate,
                        "Rate":rate,
                        "Priority":row["Priority"],
                        "Inventory":row["Inventory"],
                        "Daily":row["Daily"],
                        "Cycle time":row["Cycle time"],
                        "Indent":row["Indent"]
                    }

                    group = pd.concat([group,pd.DataFrame([new_row])])

                    remaining -= CONFIG["MIN_RUN_HOURS"]
                    break

        elif remaining > 0:

            extra = remaining / len(group)

            group["Run_Hours"] += extra
            group["Production_Qty"] = group["Run_Hours"] * group["Rate"]

        adjusted.append(group)

    return pd.concat(adjusted, ignore_index=True)

# ============================
# FINAL UTILIZATION PASS
# ============================

def final_fill_to_capacity(plan_df):

    adjusted = []

    for machine, group in plan_df.groupby("Machine"):

        used = group["Run_Hours"].sum()
        remaining = CONFIG["AVAILABLE_HOURS"] - used

        if remaining > 0:

            extra = remaining / len(group)

            group["Run_Hours"] += extra
            group["Production_Qty"] = group["Run_Hours"] * group["Rate"]

            logging.info(
                f"{machine} extended by {remaining:.2f} hours"
            )

        adjusted.append(group)

    return pd.concat(adjusted, ignore_index=True)

# ============================
# SEQUENCING
# ============================

def sequence_parts(group):

    group["Time_To_Manufacture"] = group["Indent"] / group["Rate"]

    group["Sequence_Score"] = (
        group["Priority"] * 3
        + (1/(group["Inventory"]+1))*2
        + group["Daily"]*1.5
        + (1/(group["Cycle time"]+1))
        + group["Time_To_Manufacture"]*0.5
    )

    group = group.sort_values("Sequence_Score",ascending=False)

    group["Sequence"] = range(1,len(group)+1)

    return group

# ============================
# RUN APS
# ============================

def run_dynamic_aps(df,matrix):

    df["Rate"] = (3600/df["Cycle time"]) * df["Cavity"]

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}

    df["Priority"] = df["Category"].map(priority_map)

    plan = []

    for machine, group in df.groupby("Machine"):

        remaining = CONFIG["AVAILABLE_HOURS"]

        for _, row in group.iterrows():

            if remaining <= 0:
                break

            qty, cost = optimize_part(row)

            hours = qty / row["Rate"]

            if hours > 0:

                hours = max(hours, CONFIG["MIN_RUN_HOURS"])

                if hours <= remaining:

                    plan.append({
                        "Part":row["Part"],
                        "Machine":machine,
                        "Run_Hours":hours,
                        "Production_Qty":hours*row["Rate"],
                        "Rate":row["Rate"],
                        "Priority":row["Priority"],
                        "Inventory":row["Inventory"],
                        "Daily":row["Daily"],
                        "Cycle time":row["Cycle time"],
                        "Indent":row["Indent"]
                    })

                    remaining -= hours

    final_df = pd.DataFrame(plan)

    final_df = balance_using_matrix(final_df,matrix)

    final_df = utilize_remaining_time(final_df,df,matrix)

    final_df = final_fill_to_capacity(final_df)

    final_df = final_df.groupby("Machine",group_keys=False).apply(sequence_parts)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df,machine_summary

# ============================
# RUN APS
# ============================

hz_plan,hz_machine = run_dynamic_aps(hz_long,hz_matrix)
vt_plan,vt_machine = run_dynamic_aps(vt_long,vt_matrix)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Dynamic_Hybrid_Plan_Simulation.xlsx",engine="xlsxwriter") as writer:

    hz_plan.to_excel(writer,"HZ_Plan",index=False)
    vt_plan.to_excel(writer,"VT_Plan",index=False)

    hz_machine.to_excel(writer,"HZ_Machine_Load",index=False)
    vt_machine.to_excel(writer,"VT_Machine_Load",index=False)

logging.info("APS Dynamic Planner completed successfully")

In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# CONFIGURATION
# ============================

CONFIG = {
    "SIMULATIONS":1000,
    "SHORTAGE_PENALTY":10,
    "HOLDING_PENALTY":1,
    "AVAILABLE_HOURS":22,
    "SETUP_HOURS":40/60,
    "MIN_RUN_HOURS":4,
    "MOVE_THRESHOLD":2,
    "SURVIVAL_THRESHOLD":2
}

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

logging.basicConfig(level=logging.INFO)

# ============================
# LOAD FILES
# ============================

hz = pd.read_excel(file_path,"HZ")
vt = pd.read_excel(file_path,"VT")
stats = pd.read_excel(file_path,"Sheet2")
daily = pd.read_excel(daily_path,"Sheet1")

hz_matrix = pd.read_excel(matrix_path,"HZ_Matrix").set_index("Part")
vt_matrix = pd.read_excel(matrix_path,"VT_Matrix").set_index("Part")

# ============================
# EXPAND MACHINES
# ============================

def expand_machines(df):

    base_cols=[
        "Part","Inventory","Indent",
        "Daily","Category","Cycle time","Cavity"
    ]

    machine_cols=[c for c in df.columns if c.startswith("Machine")]

    frames=[]

    for m in machine_cols:

        temp=df[base_cols].copy()
        temp["Machine"]=df[m]

        frames.append(temp)

    return pd.concat(frames,ignore_index=True)

# ============================
# CLEAN DATA
# ============================

def clean_data(df):

    df["Inventory"]=df["Inventory"].fillna(0)
    df["Indent"]=df["Indent"].fillna(0)
    df["Daily"]=df["Daily"].fillna(0)
    df["Cavity"]=df["Cavity"].fillna(1)
    df["Category"]=df["Category"].fillna("Stranger")

    df=df[df["Machine"].notna()]
    df=df[df["Cycle time"].notna()]
    df=df[df["Daily"]>0]

    return df

hz_long=clean_data(expand_machines(hz)).merge(stats,on="Part",how="left")
vt_long=clean_data(expand_machines(vt)).merge(stats,on="Part",how="left")

# ============================
# MONTE CARLO OPTIMIZER
# ============================

def optimize_part(row):

    inventory=row["Inventory"]
    demand=row["Daily"]
    mean=row["Mean_Actual"]
    std=row["Std_Deviation"]
    rate=row["Rate"]

    max_qty=int(row["Indent"]*1.2)

    qty_options=np.arange(0,max_qty+1,50)

    simulated=np.random.normal(mean,std,CONFIG["SIMULATIONS"])

    future_stock=inventory+qty_options[:,None]-demand

    shortage=np.maximum(0,simulated-future_stock)

    expected_shortage=shortage.mean(axis=1)

    expected_inventory=np.maximum(
        0,
        future_stock.mean(axis=1)-simulated.mean()
    )

    setup=np.where(qty_options>0,rate*CONFIG["SETUP_HOURS"],0)

    cost=(
        CONFIG["SHORTAGE_PENALTY"]*expected_shortage+
        CONFIG["HOLDING_PENALTY"]*expected_inventory+
        setup
    )

    hours=qty_options/rate

    feasible=hours<=CONFIG["AVAILABLE_HOURS"]

    if feasible.any():

        idx=np.argmin(np.where(feasible,cost,np.inf))

        return qty_options[idx],cost[idx]

    return 0,np.inf

# ============================
# MACHINE BALANCING
# ============================

def balance_using_matrix(plan_df,matrix):

    machine_hours=plan_df.groupby("Machine")["Run_Hours"].sum().to_dict()

    for idx,row in plan_df.iterrows():

        part=row["Part"]
        machine=row["Machine"]
        hours=row["Run_Hours"]

        if part not in matrix.index:
            continue

        compatible=matrix.loc[part]

        machines=compatible[compatible==1].index.tolist()

        for m in machines:

            if m==machine:
                continue

            if m not in machine_hours:
                machine_hours[m]=0

            free=CONFIG["AVAILABLE_HOURS"]-machine_hours[m]

            if free>CONFIG["MOVE_THRESHOLD"]:

                plan_df.loc[idx,"Machine"]=m

                machine_hours[machine]-=hours
                machine_hours[m]+=hours

                break

    return plan_df

# ============================
# UTILIZE REMAINING TIME
# ============================

def utilize_remaining_time(plan_df):

    adjusted=[]

    for machine,group in plan_df.groupby("Machine"):

        used=group["Run_Hours"].sum()

        remaining=CONFIG["AVAILABLE_HOURS"]-used

        if remaining>=CONFIG["MIN_RUN_HOURS"]:

            extra=remaining/len(group)

            group["Run_Hours"]+=extra
            group["Production_Qty"]=group["Run_Hours"]*group["Rate"]

        adjusted.append(group)

    return pd.concat(adjusted,ignore_index=True)

# ============================
# SEQUENCING
# ============================

def sequence_parts(group):

    group["Time_To_Manufacture"]=group["Indent"]/group["Rate"]

    group["Sequence_Score"]=(
        group["Priority"]*3
        +(1/(group["Inventory"]+1))*2
        +group["Daily"]*1.5
        +(1/(group["Cycle time"]+1))
        +group["Time_To_Manufacture"]*0.5
    )

    group=group.sort_values("Sequence_Score",ascending=False)

    group["Sequence"]=range(1,len(group)+1)

    return group

# ============================
# RUN APS
# ============================

def run_dynamic_aps(df,matrix):

    rejected=[]

    df["Rate"]=(3600/df["Cycle time"])*df["Cavity"]

    df["SurvivalDays"]=df["Inventory"]/df["Daily"]

    priority_map={"Runner":3,"Repeater":2,"Stranger":1}

    df["Priority"]=df["Category"].map(priority_map)

    df=df.sort_values("SurvivalDays")

    plan=[]

    for machine,group in df.groupby("Machine"):

        remaining=CONFIG["AVAILABLE_HOURS"]

        for _,row in group.iterrows():

            part=row["Part"]

            survival=row["SurvivalDays"]

            if remaining<=0:

                if survival>=CONFIG["SURVIVAL_THRESHOLD"]:

                    rejected.append({
                        "Part":part,
                        "Machine":machine,
                        "Reason":"Inventory sufficient for survival days"
                    })

                else:

                    rejected.append({
                        "Part":part,
                        "Machine":machine,
                        "Reason":"Machine capacity full"
                    })

                continue

            qty,cost=optimize_part(row)

            if qty==0:

                rejected.append({
                    "Part":part,
                    "Machine":machine,
                    "Reason":"Optimizer returned zero quantity"
                })

                continue

            hours=qty/row["Rate"]

            hours=max(hours,CONFIG["MIN_RUN_HOURS"])

            if hours>remaining:

                rejected.append({
                    "Part":part,
                    "Machine":machine,
                    "Reason":"Production time exceeds remaining machine hours"
                })

                continue

            plan.append({
                "Part":part,
                "Machine":machine,
                "Run_Hours":hours,
                "Production_Qty":hours*row["Rate"],
                "Rate":row["Rate"],
                "Priority":row["Priority"],
                "Inventory":row["Inventory"],
                "Daily":row["Daily"],
                "Cycle time":row["Cycle time"],
                "Indent":row["Indent"],
                "SurvivalDays":survival
            })

            remaining-=hours

    final_df=pd.DataFrame(plan)

    final_df=balance_using_matrix(final_df,matrix)

    final_df=utilize_remaining_time(final_df)

    final_df=final_df.groupby("Machine",group_keys=False).apply(sequence_parts)

    machine_summary=final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    rejected_df=pd.DataFrame(rejected)

    return final_df,machine_summary,rejected_df

# ============================
# RUN APS
# ============================

hz_plan,hz_machine,hz_rejected=run_dynamic_aps(hz_long,hz_matrix)
vt_plan,vt_machine,vt_rejected=run_dynamic_aps(vt_long,vt_matrix)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Dynamic_Hybrid_Plan_Simulation1.xlsx") as writer:

    hz_plan.to_excel(writer,"HZ_Plan",index=False)
    vt_plan.to_excel(writer,"VT_Plan",index=False)

    hz_machine.to_excel(writer,"HZ_Machine_Load",index=False)
    vt_machine.to_excel(writer,"VT_Machine_Load",index=False)

    hz_rejected.to_excel(writer,"HZ_Not_Planned",index=False)
    vt_rejected.to_excel(writer,"VT_Not_Planned",index=False)

logging.info("APS Dynamic Planner completed successfully")

In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# CONFIGURATION
# ============================

CONFIG = {
    "SIMULATIONS":1000,
    "SHORTAGE_PENALTY":10,
    "HOLDING_PENALTY":1,
    "AVAILABLE_HOURS":22,
    "SETUP_HOURS":40/60,
    "MIN_RUN_HOURS":4,
    "MOVE_THRESHOLD":2,
    "SURVIVAL_THRESHOLD":2
}

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

logging.basicConfig(level=logging.INFO)

# ============================
# LOAD DATA
# ============================

hz = pd.read_excel(file_path,"HZ")
vt = pd.read_excel(file_path,"VT")
stats = pd.read_excel(file_path,"Sheet2")
daily = pd.read_excel(daily_path,"Sheet1")

hz_matrix = pd.read_excel(matrix_path,"HZ_Matrix").set_index("Part")
vt_matrix = pd.read_excel(matrix_path,"VT_Matrix").set_index("Part")

# ============================
# EXPAND MACHINES
# ============================

def expand_machines(df):

    base_cols=[
        "Part","Inventory","Indent",
        "Daily","Category","Cycle time","Cavity"
    ]

    machine_cols=[c for c in df.columns if c.startswith("Machine")]

    frames=[]

    for m in machine_cols:

        temp=df[base_cols].copy()
        temp["Machine"]=df[m]

        frames.append(temp)

    return pd.concat(frames,ignore_index=True)

# ============================
# CLEAN DATA
# ============================

def clean_data(df):

    df["Inventory"]=df["Inventory"].fillna(0)
    df["Indent"]=df["Indent"].fillna(0)
    df["Daily"]=df["Daily"].fillna(0)
    df["Cavity"]=df["Cavity"].fillna(1)
    df["Category"]=df["Category"].fillna("Stranger")

    df=df[df["Machine"].notna()]
    df=df[df["Cycle time"].notna()]
    df=df[df["Daily"]>0]

    return df

hz_long=clean_data(expand_machines(hz)).merge(stats,on="Part",how="left")
vt_long=clean_data(expand_machines(vt)).merge(stats,on="Part",how="left")

# ============================
# MONTE CARLO OPTIMIZER
# ============================

def optimize_part(row):

    inventory=row["Inventory"]
    demand=row["Daily"]
    mean=row["Mean_Actual"]
    std=row["Std_Deviation"]
    rate=row["Rate"]

    max_qty=int(row["Indent"]*1.2)

    qty_options=np.arange(0,max_qty+1,50)

    simulated=np.random.normal(mean,std,CONFIG["SIMULATIONS"])

    future_stock=inventory+qty_options[:,None]-demand

    shortage=np.maximum(0,simulated-future_stock)

    expected_shortage=shortage.mean(axis=1)

    expected_inventory=np.maximum(
        0,
        future_stock.mean(axis=1)-simulated.mean()
    )

    setup=np.where(qty_options>0,rate*CONFIG["SETUP_HOURS"],0)

    cost=(
        CONFIG["SHORTAGE_PENALTY"]*expected_shortage+
        CONFIG["HOLDING_PENALTY"]*expected_inventory+
        setup
    )

    hours=qty_options/rate

    feasible=hours<=CONFIG["AVAILABLE_HOURS"]

    if feasible.any():

        idx=np.argmin(np.where(feasible,cost,np.inf))

        return qty_options[idx],cost[idx]

    return 0,np.inf

# ============================
# MACHINE BALANCING
# ============================

def balance_using_matrix(plan_df,matrix):

    machine_hours=plan_df.groupby("Machine")["Run_Hours"].sum().to_dict()

    for idx,row in plan_df.iterrows():

        part=row["Part"]
        machine=row["Machine"]
        hours=row["Run_Hours"]

        if part not in matrix.index:
            continue

        compatible=matrix.loc[part]

        machines=compatible[compatible==1].index.tolist()

        for m in machines:

            if m==machine:
                continue

            if m not in machine_hours:
                machine_hours[m]=0

            free=CONFIG["AVAILABLE_HOURS"]-machine_hours[m]

            if free>CONFIG["MOVE_THRESHOLD"]:

                plan_df.loc[idx,"Machine"]=m

                machine_hours[machine]-=hours
                machine_hours[m]+=hours

                break

    return plan_df

# ============================
# UTILIZE REMAINING TIME
# ============================

def utilize_remaining_time(plan_df):

    adjusted=[]

    for machine,group in plan_df.groupby("Machine"):

        used=group["Run_Hours"].sum()

        remaining=CONFIG["AVAILABLE_HOURS"]-used

        if remaining>0:

            extra=remaining/len(group)

            group["Run_Hours"]+=extra
            group["Production_Qty"]=group["Run_Hours"]*group["Rate"]

        adjusted.append(group)

    return pd.concat(adjusted,ignore_index=True)

# ============================
# SEQUENCING
# ============================

def sequence_parts(group):

    group["Time_To_Manufacture"]=group["Indent"]/group["Rate"]

    group["Sequence_Score"]=(
        group["Priority"]*3
        +(1/(group["Inventory"]+1))*2
        +group["Daily"]*1.5
        +(1/(group["Cycle time"]+1))
        +group["Time_To_Manufacture"]*0.5
    )

    group=group.sort_values("Sequence_Score",ascending=False)

    group["Sequence"]=range(1,len(group)+1)

    return group

# ============================
# RUN APS
# ============================

def run_dynamic_aps(df,matrix):

    rejected=[]

    df["Rate"]=(3600/df["Cycle time"])*df["Cavity"]

    df["SurvivalDays"]=df["Inventory"]/df["Daily"]

    priority_map={"Runner":3,"Repeater":2,"Stranger":1}

    df["Priority"]=df["Category"].map(priority_map)

    df=df.sort_values("SurvivalDays")

    plan=[]

    for machine,group in df.groupby("Machine"):

        remaining=CONFIG["AVAILABLE_HOURS"]

        for _,row in group.iterrows():

            part=row["Part"]

            survival=row["SurvivalDays"]

            if remaining<=0:

                rejected.append({
                    "Part":part,
                    "Reason":"Machine capacity full"
                })

                continue

            qty,cost=optimize_part(row)

            if qty==0:

                rejected.append({
                    "Part":part,
                    "Reason":"Optimizer returned zero quantity"
                })

                continue

            hours=qty/row["Rate"]

            hours=max(hours,CONFIG["MIN_RUN_HOURS"])

            if hours>remaining:

                rejected.append({
                    "Part":part,
                    "Reason":"Production exceeds remaining hours"
                })

                continue

            plan.append({
                "Part":part,
                "Machine":machine,
                "Run_Hours":hours,
                "Production_Qty":hours*row["Rate"],
                "Rate":row["Rate"],
                "Priority":row["Priority"],
                "Inventory":row["Inventory"],
                "Daily":row["Daily"],
                "Cycle time":row["Cycle time"],
                "Indent":row["Indent"],
                "SurvivalDays":survival
            })

            remaining-=hours

    final_df=pd.DataFrame(plan)

    final_df=balance_using_matrix(final_df,matrix)

    final_df=utilize_remaining_time(final_df)

    final_df=final_df.groupby("Machine",group_keys=False).apply(sequence_parts)

    machine_summary=final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    planned_parts=set(final_df["Part"])
    original_parts=set(df["Part"])

    missing_parts=original_parts-planned_parts

    filtered=[r for r in rejected if r["Part"] in missing_parts]

    rejected_df=pd.DataFrame(filtered).drop_duplicates("Part")

    return final_df,machine_summary,rejected_df

# ============================
# RUN APS
# ============================

hz_plan,hz_machine,hz_rejected=run_dynamic_aps(hz_long,hz_matrix)
vt_plan,vt_machine,vt_rejected=run_dynamic_aps(vt_long,vt_matrix)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Dynamic_Hybrid_Plan_Simulation.xlsx") as writer:

    hz_plan.to_excel(writer,"HZ_Plan",index=False)
    vt_plan.to_excel(writer,"VT_Plan",index=False)

    hz_machine.to_excel(writer,"HZ_Machine_Load",index=False)
    vt_machine.to_excel(writer,"VT_Machine_Load",index=False)

    hz_rejected.to_excel(writer,"HZ_Not_Planned",index=False)
    vt_rejected.to_excel(writer,"VT_Not_Planned",index=False)

logging.info("APS Dynamic Planner completed successfully")

In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# CONFIGURATION
# ============================

CONFIG = {
    "SIMULATIONS": 1000,
    "SHORTAGE_PENALTY": 10,
    "HOLDING_PENALTY": 1,
    "AVAILABLE_HOURS": 22,
    "SETUP_HOURS": 40 / 60,
    "MIN_RUN_HOURS": 4,
    "MOVE_THRESHOLD": 2
}

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx"
matrix_path = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================
# LOAD FILES
# ============================

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")
stats = pd.read_excel(file_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix").set_index("Part")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix").set_index("Part")

# ============================
# EXPAND MACHINES
# ============================

def expand_machines(df):

    base_cols = [
        "Part","Inventory","Indent",
        "Daily","Category","Cycle time","Cavity"
    ]

    machine_cols = [c for c in df.columns if c.startswith("Machine")]

    frames = []

    for m in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[m]
        frames.append(temp)

    return pd.concat(frames, ignore_index=True)

# ============================
# CLEAN DATA
# ============================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Indent"] = df["Indent"].fillna(0)
    df["Daily"] = df["Daily"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Cycle time"].notna()]
    df = df[df["Daily"] > 0]

    return df

hz_long = clean_data(expand_machines(hz)).merge(stats, on="Part", how="left")
vt_long = clean_data(expand_machines(vt)).merge(stats, on="Part", how="left")

# ============================
# MONTE CARLO OPTIMIZER
# ============================

def optimize_part(row):

    inventory = row["Inventory"]
    demand = row["Daily"]
    mean = row["Mean_Actual"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    max_qty = int(row["Indent"] * 1.2)

    qty_options = np.arange(0, max_qty + 1, 50)

    simulated = np.random.normal(mean, std, CONFIG["SIMULATIONS"])

    future_stock = inventory + qty_options[:, None] - demand

    shortage = np.maximum(0, simulated - future_stock)

    expected_shortage = shortage.mean(axis=1)

    expected_inventory = np.maximum(
        0,
        future_stock.mean(axis=1) - simulated.mean()
    )

    setup_penalty = np.where(
        qty_options > 0,
        rate * CONFIG["SETUP_HOURS"],
        0
    )

    cost = (
        CONFIG["SHORTAGE_PENALTY"] * expected_shortage +
        CONFIG["HOLDING_PENALTY"] * expected_inventory +
        setup_penalty
    )

    hours_needed = qty_options / rate

    feasible = hours_needed <= CONFIG["AVAILABLE_HOURS"]

    if feasible.any():
        best_idx = np.argmin(np.where(feasible, cost, np.inf))
        return qty_options[best_idx], cost[best_idx]

    return 0, np.inf

# ============================
# MACHINE BALANCING
# ============================

def balance_using_matrix(plan_df, matrix):

    machine_hours = plan_df.groupby("Machine")["Run_Hours"].sum().to_dict()

    for idx,row in plan_df.iterrows():

        part = row["Part"]
        machine = row["Machine"]
        hours = row["Run_Hours"]

        if part not in matrix.index:
            continue

        if machine_hours[machine] <= CONFIG["AVAILABLE_HOURS"]:
            continue

        compatible = matrix.loc[part]
        machines = compatible[compatible == 1].index.tolist()

        for m in machines:

            if m == machine:
                continue

            if m not in machine_hours:
                machine_hours[m] = 0

            free = CONFIG["AVAILABLE_HOURS"] - machine_hours[m]

            if free > CONFIG["MOVE_THRESHOLD"]:

                plan_df.loc[idx,"Machine"] = m

                machine_hours[machine] -= hours
                machine_hours[m] += hours

                break

    return plan_df

# ============================
# UTILIZE REMAINING TIME
# ============================

def utilize_remaining_time(plan_df, df, matrix):

    adjusted = []

    for machine, group in plan_df.groupby("Machine"):

        used = group["Run_Hours"].sum()
        remaining = CONFIG["AVAILABLE_HOURS"] - used

        if remaining >= CONFIG["MIN_RUN_HOURS"]:

            possible_parts = df[df["Machine"] == machine]

            for _, row in possible_parts.iterrows():

                part = row["Part"]

                if part in matrix.index and machine in matrix.columns:
                    if matrix.loc[part, machine] != 1:
                        continue

                rate = row["Rate"]

                if CONFIG["MIN_RUN_HOURS"] <= remaining:

                    new_row = {
                        "Part":part,
                        "Machine":machine,
                        "Run_Hours":CONFIG["MIN_RUN_HOURS"],
                        "Production_Qty":CONFIG["MIN_RUN_HOURS"]*rate,
                        "Rate":rate,
                        "Priority":row["Priority"],
                        "Inventory":row["Inventory"],
                        "Daily":row["Daily"],
                        "Cycle time":row["Cycle time"],
                        "Indent":row["Indent"]
                    }

                    group = pd.concat([group,pd.DataFrame([new_row])])

                    remaining -= CONFIG["MIN_RUN_HOURS"]
                    break

        elif remaining > 0:

            extra = remaining / len(group)

            group["Run_Hours"] += extra
            group["Production_Qty"] = group["Run_Hours"] * group["Rate"]

        adjusted.append(group)

    return pd.concat(adjusted, ignore_index=True)

# ============================
# FINAL UTILIZATION PASS
# ============================

def final_fill_to_capacity(plan_df):

    adjusted = []

    for machine, group in plan_df.groupby("Machine"):

        used = group["Run_Hours"].sum()
        remaining = CONFIG["AVAILABLE_HOURS"] - used

        if remaining > 0:

            extra = remaining / len(group)

            group["Run_Hours"] += extra
            group["Production_Qty"] = group["Run_Hours"] * group["Rate"]

        adjusted.append(group)

    return pd.concat(adjusted, ignore_index=True)

# ============================
# SEQUENCING
# ============================

def sequence_parts(group):

    group["Time_To_Manufacture"] = group["Indent"] / group["Rate"]

    group["Sequence_Score"] = (
        group["Priority"] * 3
        + (1/(group["Inventory"]+1))*2
        + group["Daily"]*1.5
        + (1/(group["Cycle time"]+1))
        + group["Time_To_Manufacture"]*0.5
    )

    group = group.sort_values("Sequence_Score",ascending=False)

    group["Sequence"] = range(1,len(group)+1)

    return group

# ============================
# RUN APS
# ============================

def run_dynamic_aps(df,matrix):

    df["Rate"] = (3600/df["Cycle time"]) * df["Cavity"]

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}

    df["Priority"] = df["Category"].map(priority_map)

    plan = []

    for machine, group in df.groupby("Machine"):

        remaining = CONFIG["AVAILABLE_HOURS"]

        for _, row in group.iterrows():

            if remaining <= 0:
                break

            qty, cost = optimize_part(row)

            hours = qty / row["Rate"]

            if hours > 0:

                hours = max(hours, CONFIG["MIN_RUN_HOURS"])

                if hours <= remaining:

                    plan.append({
                        "Part":row["Part"],
                        "Machine":machine,
                        "Run_Hours":hours,
                        "Production_Qty":hours*row["Rate"],
                        "Rate":row["Rate"],
                        "Priority":row["Priority"],
                        "Inventory":row["Inventory"],
                        "Daily":row["Daily"],
                        "Cycle time":row["Cycle time"],
                        "Indent":row["Indent"]
                    })

                    remaining -= hours

    final_df = pd.DataFrame(plan)

    final_df = balance_using_matrix(final_df,matrix)

    final_df = utilize_remaining_time(final_df,df,matrix)

    final_df = final_fill_to_capacity(final_df)

    final_df = final_df.groupby("Machine",group_keys=False).apply(sequence_parts)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df,machine_summary

# ============================
# DIAGNOSTICS FOR MISSING PARTS
# ============================

def diagnose_missing_parts(original_df, plan_df, matrix):

    planned_parts = set(plan_df["Part"])
    original_parts = set(original_df["Part"])

    missing = original_parts - planned_parts

    diagnostics = []

    for part in missing:

        rows = original_df[original_df["Part"] == part]

        reason = "Machine capacity used by other parts"

        for _, row in rows.iterrows():

            machine = row["Machine"]

            if part not in matrix.index:
                reason = "Part not in compatibility matrix"
                break

            if machine not in matrix.columns:
                reason = "Machine not in compatibility matrix"
                break

            if matrix.loc[part, machine] != 1:
                reason = "Machine not compatible"
                continue

            rate = (3600 / row["Cycle time"]) * row["Cavity"]

            if pd.isna(rate) or rate == 0:
                reason = "Rate cannot be calculated"
                break

            qty, cost = optimize_part(row)

            if qty == 0:
                reason = "Optimizer returned zero quantity"
                break

        diagnostics.append({
            "Part":part,
            "Reason_Not_Planned":reason
        })

    return pd.DataFrame(diagnostics)

# ============================
# RUN APS
# ============================

hz_plan,hz_machine = run_dynamic_aps(hz_long,hz_matrix)
vt_plan,vt_machine = run_dynamic_aps(vt_long,vt_matrix)

hz_diag = diagnose_missing_parts(hz_long,hz_plan,hz_matrix)
vt_diag = diagnose_missing_parts(vt_long,vt_plan,vt_matrix)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Dynamic_Hybrid_Plan_Simulation.xlsx",engine="xlsxwriter") as writer:

    hz_plan.to_excel(writer,"HZ_Plan",index=False)
    vt_plan.to_excel(writer,"VT_Plan",index=False)

    hz_machine.to_excel(writer,"HZ_Machine_Load",index=False)
    vt_machine.to_excel(writer,"VT_Machine_Load",index=False)

    hz_diag.to_excel(writer,"HZ_Not_Planned",index=False)
    vt_diag.to_excel(writer,"VT_Not_Planned",index=False)

logging.info("APS Dynamic Planner completed successfully")

In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# CONFIGURATION
# ============================

CONFIG = {
    "SIMULATIONS": 1000,
    "SHORTAGE_PENALTY": 10,
    "HOLDING_PENALTY": 1,
    "AVAILABLE_HOURS": 22,
    "SETUP_HOURS": 40 / 60,
    "MIN_RUN_HOURS": 4,
    "MOVE_THRESHOLD": 2
}

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx"
matrix_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================
# LOAD FILES
# ============================

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")
stats = pd.read_excel(file_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

hz_matrix = pd.read_excel(matrix_path, sheet_name="HZ_Matrix").set_index("Part")
vt_matrix = pd.read_excel(matrix_path, sheet_name="VT_Matrix").set_index("Part")

# ============================
# MACHINE SELECTION (PRIMARY)
# ============================

def expand_machines(df):

    base_cols = [
        "Part","Inventory","Indent",
        "Daily","Category","Cycle time","Cavity"
    ]

    machine_cols = [c for c in df.columns if c.startswith("Machine")]

    df["Machine"] = df[machine_cols].bfill(axis=1).iloc[:,0]

    return df[base_cols + ["Machine"]]

# ============================
# CLEAN DATA
# ============================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Indent"] = df["Indent"].fillna(0)
    df["Daily"] = df["Daily"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Cycle time"].notna()]
    df = df[df["Daily"] > 0]

    return df

hz_long = clean_data(expand_machines(hz)).merge(stats, on="Part", how="left")
vt_long = clean_data(expand_machines(vt)).merge(stats, on="Part", how="left")

# ============================
# MONTE CARLO OPTIMIZER
# ============================

def optimize_part(row):

    inventory = row["Inventory"]
    demand = row["Daily"]
    mean = row["Mean_Actual"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    max_qty = int(row["Indent"] * 1.2)

    qty_options = np.arange(0, max_qty + 1, 50)

    simulated = np.random.normal(mean, std, CONFIG["SIMULATIONS"])

    future_stock = inventory + qty_options[:, None] - demand

    shortage = np.maximum(0, simulated - future_stock)

    expected_shortage = shortage.mean(axis=1)

    expected_inventory = np.maximum(
        0,
        future_stock.mean(axis=1) - simulated.mean()
    )

    setup_penalty = np.where(
        qty_options > 0,
        rate * CONFIG["SETUP_HOURS"],
        0
    )

    cost = (
        CONFIG["SHORTAGE_PENALTY"] * expected_shortage +
        CONFIG["HOLDING_PENALTY"] * expected_inventory +
        setup_penalty
    )

    hours_needed = qty_options / rate

    feasible = hours_needed <= CONFIG["AVAILABLE_HOURS"]

    if feasible.any():
        best_idx = np.argmin(np.where(feasible, cost, np.inf))
        return qty_options[best_idx], cost[best_idx]

    return 0, np.inf

# ============================
# MACHINE BALANCING
# ============================

def balance_using_matrix(plan_df, matrix):

    machine_hours = plan_df.groupby("Machine")["Run_Hours"].sum().to_dict()

    for idx,row in plan_df.iterrows():

        part = row["Part"]
        machine = row["Machine"]
        hours = row["Run_Hours"]

        if part not in matrix.index:
            continue

        if machine_hours[machine] <= CONFIG["AVAILABLE_HOURS"]:
            continue

        compatible = matrix.loc[part]
        machines = compatible[compatible == 1].index.tolist()

        for m in machines:

            if m == machine:
                continue

            if m not in machine_hours:
                machine_hours[m] = 0

            free = CONFIG["AVAILABLE_HOURS"] - machine_hours[m]

            if free > CONFIG["MOVE_THRESHOLD"]:

                plan_df.loc[idx,"Machine"] = m

                machine_hours[machine] -= hours
                machine_hours[m] += hours

                break

    return plan_df

# ============================
# UTILIZE REMAINING TIME
# ============================

def utilize_remaining_time(plan_df, df, matrix):

    adjusted = []

    for machine, group in plan_df.groupby("Machine"):

        used = group["Run_Hours"].sum()
        remaining = CONFIG["AVAILABLE_HOURS"] - used

        if remaining >= CONFIG["MIN_RUN_HOURS"]:

            possible_parts = df[df["Machine"] == machine]

            for _, row in possible_parts.iterrows():

                part = row["Part"]

                if part in matrix.index and machine in matrix.columns:

                    val = matrix.loc[part, machine]

                    if isinstance(val, pd.Series):
                        val = val.iloc[0]

                    if val != 1:
                        continue

                rate = row["Rate"]

                new_row = {
                    "Part":part,
                    "Machine":machine,
                    "Run_Hours":CONFIG["MIN_RUN_HOURS"],
                    "Production_Qty":CONFIG["MIN_RUN_HOURS"]*rate,
                    "Rate":rate,
                    "Priority":row["Priority"],
                    "Inventory":row["Inventory"],
                    "Daily":row["Daily"],
                    "Cycle time":row["Cycle time"],
                    "Indent":row["Indent"]
                }

                group = pd.concat([group,pd.DataFrame([new_row])])

                break

        elif remaining > 0:

            extra = remaining / len(group)

            group["Run_Hours"] += extra
            group["Production_Qty"] = group["Run_Hours"] * group["Rate"]

        adjusted.append(group)

    return pd.concat(adjusted, ignore_index=True)

# ============================
# FINAL UTILIZATION PASS
# ============================

def final_fill_to_capacity(plan_df):

    adjusted = []

    for machine, group in plan_df.groupby("Machine"):

        used = group["Run_Hours"].sum()
        remaining = CONFIG["AVAILABLE_HOURS"] - used

        if remaining > 0:

            extra = remaining / len(group)

            group["Run_Hours"] += extra
            group["Production_Qty"] = group["Run_Hours"] * group["Rate"]

        adjusted.append(group)

    return pd.concat(adjusted, ignore_index=True)

# ============================
# SEQUENCING
# ============================

def sequence_parts(group):

    group["Time_To_Manufacture"] = group["Indent"] / group["Rate"]

    group["Sequence_Score"] = (
        group["Priority"] * 3
        + (1/(group["Inventory"]+1))*2
        + group["Daily"]*1.5
        + (1/(group["Cycle time"]+1))
        + group["Time_To_Manufacture"]*0.5
    )

    group = group.sort_values("Sequence_Score",ascending=False)

    group["Sequence"] = range(1,len(group)+1)

    return group

# ============================
# RUN APS
# ============================

def run_dynamic_aps(df,matrix):

    df["Rate"] = (3600/df["Cycle time"]) * df["Cavity"]

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}

    df["Priority"] = df["Category"].map(priority_map)

    plan = []

    for machine, group in df.groupby("Machine"):

        remaining = CONFIG["AVAILABLE_HOURS"]

        for _, row in group.iterrows():

            if remaining <= 0:
                break

            qty, cost = optimize_part(row)

            hours = qty / row["Rate"]

            if hours > 0:

                hours = max(hours, CONFIG["MIN_RUN_HOURS"])

                if hours <= remaining:

                    plan.append({
                        "Part":row["Part"],
                        "Machine":machine,
                        "Run_Hours":hours,
                        "Production_Qty":hours*row["Rate"],
                        "Rate":row["Rate"],
                        "Priority":row["Priority"],
                        "Inventory":row["Inventory"],
                        "Daily":row["Daily"],
                        "Cycle time":row["Cycle time"],
                        "Indent":row["Indent"]
                    })

                    remaining -= hours

    final_df = pd.DataFrame(plan)

    final_df = balance_using_matrix(final_df,matrix)

    final_df = utilize_remaining_time(final_df,df,matrix)

    final_df = final_fill_to_capacity(final_df)

    final_df = final_df.groupby("Machine",group_keys=False).apply(sequence_parts, include_groups=False)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df,machine_summary

# ============================
# DIAGNOSTICS
# ============================

def diagnose_missing_parts(original_df, plan_df, matrix):

    planned_parts = set(plan_df["Part"])
    original_parts = set(original_df["Part"])

    missing = original_parts - planned_parts

    diagnostics = []

    for part in missing:

        diagnostics.append({
            "Part":part,
            "Reason_Not_Planned":"Machine capacity used by other parts"
        })

    return pd.DataFrame(diagnostics)

# ============================
# RUN APS
# ============================

hz_plan,hz_machine = run_dynamic_aps(hz_long,hz_matrix)
vt_plan,vt_machine = run_dynamic_aps(vt_long,vt_matrix)

hz_diag = diagnose_missing_parts(hz_long,hz_plan,hz_matrix)
vt_diag = diagnose_missing_parts(vt_long,vt_plan,vt_matrix)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Dynamic_Hybrid_Plan_Simulation1.xlsx",engine="xlsxwriter") as writer:

    hz_plan.to_excel(writer,"HZ_Plan",index=False)
    vt_plan.to_excel(writer,"VT_Plan",index=False)

    hz_machine.to_excel(writer,"HZ_Machine_Load",index=False)
    vt_machine.to_excel(writer,"VT_Machine_Load",index=False)

    hz_diag.to_excel(writer,"HZ_Not_Planned",index=False)
    vt_diag.to_excel(writer,"VT_Not_Planned",index=False)

logging.info("APS Dynamic Planner completed successfully")

In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# CONFIGURATION
# ============================

CONFIG = {
    "SIMULATIONS": 1000,
    "SHORTAGE_PENALTY": 10,
    "HOLDING_PENALTY": 1,
    "AVAILABLE_HOURS": 22,
    "SETUP_HOURS": 40 / 60,
    "MIN_RUN_HOURS": 4,
    "SMALL_PART_THRESHOLD": 6
}

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx"

# ============================
# LOGGING
# ============================

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================
# LOAD FILES
# ============================

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")
stats = pd.read_excel(file_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

# ============================
# EXPAND MACHINES
# ============================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Daily","Category","Cycle time","Cavity"]
    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]
        df_list.append(temp)

    return pd.concat(df_list, ignore_index=True)

# ============================
# CLEAN DATA
# ============================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Indent"] = df["Indent"].fillna(0)
    df["Daily"] = df["Daily"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Cycle time"].notna()]
    df = df[df["Daily"] > 0]

    return df

hz_long = clean_data(expand_machines(hz)).merge(stats, on="Part", how="left")
vt_long = clean_data(expand_machines(vt)).merge(stats, on="Part", how="left")

# ============================
# SIMULATION OPTIMIZER
# ============================

def optimize_part(row):

    inventory = row["Inventory"]
    demand_tomorrow = row["Daily"]
    tentative_future = row["Mean_Actual"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    max_qty = int(row["Indent"] * 1.2)
    qty_options = np.arange(0, max_qty + 1, 50)

    simulated_demand = np.random.normal(tentative_future, std, CONFIG["SIMULATIONS"])

    future_stock = inventory + qty_options[:, None] - demand_tomorrow

    shortage = np.maximum(0, simulated_demand - future_stock)
    expected_shortage = shortage.mean(axis=1)

    expected_inventory = np.maximum(0, future_stock.mean(axis=1) - simulated_demand.mean())

    setup_penalty = np.where(qty_options > 0, rate * CONFIG["SETUP_HOURS"], 0)

    cost = (
        CONFIG["SHORTAGE_PENALTY"] * expected_shortage
        + CONFIG["HOLDING_PENALTY"] * expected_inventory
        + setup_penalty
    )

    hours_needed = qty_options / rate

    feasible = hours_needed <= CONFIG["AVAILABLE_HOURS"]

    if feasible.any():

        best_idx = np.argmin(np.where(feasible, cost, np.inf))

        return qty_options[best_idx], cost[best_idx]

    return 0, np.inf

# ============================
# MACHINE UTILIZATION
# ============================

def adjust_machine_utilization(final_df):

    adjusted_plan = []

    for machine, group in final_df.groupby("Machine"):

        total_hours = group["Run_Hours"].sum()
        num_parts = len(group)

        if num_parts == 1 and group.iloc[0]["Run_Hours"] >= CONFIG["AVAILABLE_HOURS"]:

            group.iloc[0, group.columns.get_loc("Run_Hours")] = CONFIG["AVAILABLE_HOURS"]

            group.iloc[0, group.columns.get_loc("Production_Qty")] = round(
                group.iloc[0]["Rate"] * CONFIG["AVAILABLE_HOURS"], 0
            )

        elif num_parts == 1 and total_hours < CONFIG["AVAILABLE_HOURS"]:

            leftover = CONFIG["AVAILABLE_HOURS"] - total_hours

            logging.info(f"Machine {machine} has {leftover} idle hours")

        elif num_parts > 1 and total_hours < CONFIG["AVAILABLE_HOURS"]:

            leftover = CONFIG["AVAILABLE_HOURS"] - total_hours

            extra_per_part = leftover / num_parts

            group["Run_Hours"] += extra_per_part

            group["Production_Qty"] = (group["Run_Hours"] * group["Rate"]).round(0)

        adjusted_plan.append(group)

    return pd.concat(adjusted_plan, ignore_index=True)

# ============================
# APS ENGINE
# ============================

def run_dynamic_aps(df):

    df["Rate"] = (3600 / df["Cycle time"].replace(0, np.nan)) * df["Cavity"]

    df = df[df["Rate"].notna()]

    df["Monthly_Hours"] = df["Indent"] / df["Rate"]

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}

    df["Priority"] = df["Category"].map(lambda x: priority_map.get(x,0))

    final_plan = []
    not_planned = []

    for machine, group in df.groupby("Machine"):

        remaining_hours = CONFIG["AVAILABLE_HOURS"]

        group = group.sort_values(
            by=["Priority","Monthly_Hours"],
            ascending=[False,True]
        )

        for _, row in group.iterrows():

            planned_qty, cost = optimize_part(row)

            if planned_qty == 0:

                not_planned.append({
                    "Part":row["Part"],
                    "Machine":machine,
                    "Reason":"Optimizer suggested zero production"
                })

                continue

            hours = planned_qty / row["Rate"]

            if hours > CONFIG["AVAILABLE_HOURS"]:

                not_planned.append({
                    "Part":row["Part"],
                    "Machine":machine,
                    "Reason":"Required hours exceed machine capacity"
                })

                continue

            hours = max(hours, CONFIG["MIN_RUN_HOURS"])

            if hours > remaining_hours:

                inventory = row["Inventory"]
                today_demand = row["Daily"]

                if inventory >= today_demand:

                    reason = "Machine hours insufficient but inventory covers today's demand"

                else:

                    shortage = today_demand - inventory

                    reason = f"Machine hours insufficient and shortage risk today ({shortage} units)"

                not_planned.append({
                    "Part":row["Part"],
                    "Machine":machine,
                    "Inventory":inventory,
                    "Daily_Demand":today_demand,
                    "Reason":reason
                })

                continue

            final_plan.append({
                "Part":row["Part"],
                "Category":row["Category"],
                "Inventory":row["Inventory"],
                "Indent":row["Indent"],
                "Daily":row["Daily"],
                "Cycle time":row["Cycle time"],
                "Cavity":row["Cavity"],
                "Rate":round(row["Rate"],2),
                "Machine":machine,
                "Run_Hours":round(hours,2),
                "Production_Qty":round(hours * row["Rate"],0),
                "Cost":round(cost,2)
            })

            remaining_hours -= hours

    final_df = pd.DataFrame(final_plan)

    final_df = adjust_machine_utilization(final_df)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    not_planned_df = pd.DataFrame(not_planned)

    return final_df, machine_summary, not_planned_df


# ============================
# RUN APS
# ============================

hz_plan, hz_machine, hz_not_planned = run_dynamic_aps(hz_long)
vt_plan, vt_machine, vt_not_planned = run_dynamic_aps(vt_long)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Dynamic_Hybrid_Plan_Simulation2.xlsx", engine="xlsxwriter") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

    hz_not_planned.to_excel(writer, sheet_name="HZ_Not_Planned", index=False)
    vt_not_planned.to_excel(writer, sheet_name="VT_Not_Planned", index=False)

logging.info("APS Optimization + Not Planned Analysis Complete")

In [ ]:
import pandas as pd
import numpy as np
import logging

# ============================
# CONFIGURATION
# ============================

CONFIG = {
    "SIMULATIONS": 1000,
    "SHORTAGE_PENALTY": 10,
    "HOLDING_PENALTY": 1,
    "AVAILABLE_HOURS": 22,
    "SETUP_HOURS": 40 / 60,
    "MIN_RUN_HOURS": 4,
    "SMALL_PART_THRESHOLD": 6
}

file_path = "C:/Users/Ex0164/Book1.xlsx"
daily_path = "C:/Users/Ex0164/Important codes/Child_for_28feb_actual.xlsx"

# ============================
# LOGGING
# ============================

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ============================
# LOAD FILES
# ============================

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")
stats = pd.read_excel(file_path, sheet_name="Sheet2")
daily = pd.read_excel(daily_path, sheet_name="Sheet1")

# ============================
# EXPAND MACHINES
# ============================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Daily","Category","Cycle time","Cavity"]
    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]
        df_list.append(temp)

    return pd.concat(df_list, ignore_index=True)

# ============================
# CLEAN DATA
# ============================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Indent"] = df["Indent"].fillna(0)
    df["Daily"] = df["Daily"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Cycle time"].notna()]
    df = df[df["Daily"] > 0]

    return df

hz_long = clean_data(expand_machines(hz)).merge(stats, on="Part", how="left")
vt_long = clean_data(expand_machines(vt)).merge(stats, on="Part", how="left")

# ============================
# SIMULATION OPTIMIZER
# ============================

def optimize_part(row):

    inventory = row["Inventory"]
    demand_tomorrow = row["Daily"]
    tentative_future = row["Mean_Actual"]
    std = row["Std_Deviation"]
    rate = row["Rate"]

    max_qty = int(row["Indent"] * 1.2)
    qty_options = np.arange(0, max_qty + 1, 50)

    simulated_demand = np.random.normal(tentative_future, std, CONFIG["SIMULATIONS"])

    future_stock = inventory + qty_options[:, None] - demand_tomorrow

    shortage = np.maximum(0, simulated_demand - future_stock)
    expected_shortage = shortage.mean(axis=1)

    expected_inventory = np.maximum(0, future_stock.mean(axis=1) - simulated_demand.mean())

    setup_penalty = np.where(qty_options > 0, rate * CONFIG["SETUP_HOURS"], 0)

    cost = (
        CONFIG["SHORTAGE_PENALTY"] * expected_shortage
        + CONFIG["HOLDING_PENALTY"] * expected_inventory
        + setup_penalty
    )

    hours_needed = qty_options / rate

    feasible = hours_needed <= CONFIG["AVAILABLE_HOURS"]

    if feasible.any():

        best_idx = np.argmin(np.where(feasible, cost, np.inf))

        return qty_options[best_idx], cost[best_idx]

    return 0, np.inf

# ============================
# MACHINE UTILIZATION
# ============================

def adjust_machine_utilization(final_df):

    adjusted_plan = []

    for machine, group in final_df.groupby("Machine"):

        total_hours = group["Run_Hours"].sum()
        num_parts = len(group)

        if num_parts == 1 and group.iloc[0]["Run_Hours"] >= CONFIG["AVAILABLE_HOURS"]:

            group.iloc[0, group.columns.get_loc("Run_Hours")] = CONFIG["AVAILABLE_HOURS"]

            group.iloc[0, group.columns.get_loc("Production_Qty")] = round(
                group.iloc[0]["Rate"] * CONFIG["AVAILABLE_HOURS"], 0
            )

        elif num_parts == 1 and total_hours < CONFIG["AVAILABLE_HOURS"]:

            leftover = CONFIG["AVAILABLE_HOURS"] - total_hours
            logging.info(f"Machine {machine} has {leftover} idle hours")

        elif num_parts > 1 and total_hours < CONFIG["AVAILABLE_HOURS"]:

            leftover = CONFIG["AVAILABLE_HOURS"] - total_hours
            extra_per_part = leftover / num_parts

            group["Run_Hours"] += extra_per_part
            group["Production_Qty"] = (group["Run_Hours"] * group["Rate"]).round(0)

        adjusted_plan.append(group)

    return pd.concat(adjusted_plan, ignore_index=True)

# ============================
# APS ENGINE
# ============================

def run_dynamic_aps(df, unique_part=False):

    df["Rate"] = (3600 / df["Cycle time"].replace(0, np.nan)) * df["Cavity"]
    df = df[df["Rate"].notna()]

    df["Monthly_Hours"] = df["Indent"] / df["Rate"]

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
    df["Priority"] = df["Category"].map(lambda x: priority_map.get(x,0))

    final_plan = []
    not_planned = []

    scheduled_parts = set()

    for machine, group in df.groupby("Machine"):

        remaining_hours = CONFIG["AVAILABLE_HOURS"]

        group = group.sort_values(
            by=["Priority","Monthly_Hours"],
            ascending=[False,True]
        )

        for _, row in group.iterrows():

            part = row["Part"]

            if unique_part and part in scheduled_parts:
                continue

            planned_qty, cost = optimize_part(row)

            if planned_qty == 0:

                not_planned.append({
                    "Part":part,
                    "Machine":machine,
                    "Reason":"Optimizer suggested zero production"
                })

                continue

            hours = planned_qty / row["Rate"]

            if hours > CONFIG["AVAILABLE_HOURS"]:

                not_planned.append({
                    "Part":part,
                    "Machine":machine,
                    "Reason":"Required hours exceed machine capacity"
                })

                continue

            hours = max(hours, CONFIG["MIN_RUN_HOURS"])

            if hours > remaining_hours:

                inventory = row["Inventory"]
                today_demand = row["Daily"]

                if inventory >= today_demand:

                    reason = "Machine hours insufficient but inventory covers today's demand"

                else:

                    shortage = today_demand - inventory
                    reason = f"Machine hours insufficient and shortage risk today ({shortage} units)"

                not_planned.append({
                    "Part":part,
                    "Machine":machine,
                    "Inventory":inventory,
                    "Daily_Demand":today_demand,
                    "Reason":reason
                })

                continue

            final_plan.append({
                "Part":part,
                "Category":row["Category"],
                "Inventory":row["Inventory"],
                "Indent":row["Indent"],
                "Daily":row["Daily"],
                "Cycle time":row["Cycle time"],
                "Cavity":row["Cavity"],
                "Rate":round(row["Rate"],2),
                "Machine":machine,
                "Run_Hours":round(hours,2),
                "Production_Qty":round(hours * row["Rate"],0),
                "Cost":round(cost,2)
            })

            remaining_hours -= hours

            if unique_part:
                scheduled_parts.add(part)

    final_df = pd.DataFrame(final_plan)

    final_df = adjust_machine_utilization(final_df)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    not_planned_df = pd.DataFrame(not_planned)

    return final_df, machine_summary, not_planned_df


# ============================
# RUN APS
# ============================

hz_plan, hz_machine, hz_not_planned = run_dynamic_aps(hz_long, unique_part=True)
vt_plan, vt_machine, vt_not_planned = run_dynamic_aps(vt_long, unique_part=False)

# ============================
# SAVE OUTPUT
# ============================

with pd.ExcelWriter("APS_Dynamic_Hybrid_Plan_Simulation2.xlsx", engine="xlsxwriter") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

    hz_not_planned.to_excel(writer, sheet_name="HZ_Not_Planned", index=False)
    vt_not_planned.to_excel(writer, sheet_name="VT_Not_Planned", index=False)

logging.info("APS Optimization + Compatibility Matrix Logic Applied")